## Load data

### Load plaintexts

In [ ]:
# Initialize an array to store the extracted values
hex_arrays = []
path="/media/abish/Extreme Pro/Sbox_600/"#sbox_600/"#"courses/sca101/res/"
# Open the file and process each line
with open(path+"plaintexts.txt", "r") as file:
    for line in file:
        # Split the line by spaces and extract the hex values (ignoring the label before the colon)
        parts = line.strip().split(":")
        if len(parts) == 2:
            hex_values = parts[1].strip().split()
            hex_arrays.append(hex_values)

# Print the resulting array
## for idx, hex_array in enu|merate(hex_arrays, start=1):
   ## print(f"vmem{idx}: {hex_array}")

### Convert data + remove last value of traces

In [ ]:
import numpy as np
import os
trace_arrayx= np.array(data_arrays)[:, :-1] #remove last value of traces
number_of_traces =600
textin_arraycc=hex_arrays[:number_of_traces]
textin_array = [[int(value, 16) for value in hex_array] for hex_array in textin_arraycc] # convert to hex
print(f"textin_array array shape: {np.shape(textin_array)}")
print(f"trace_arrayx array shape: {np.shape(trace_arrayx)}")

## CPA

In [ ]:
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from Crypto.Cipher import AES  # PyCryptodome library for AES verification
import time


sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

def aes_internal(inputdata, key):
    return sbox[inputdata ^ key]

HW = [bin(n).count("1") for n in range(0, 256)]


def mean(X):
    return np.sum(X, axis=0)/len(X)

def std_dev(X, X_bar):
    return np.sqrt(np.sum((X-X_bar)**2, axis=0))

def cov(X, X_bar, Y, Y_bar):
    return np.sum((X-X_bar)*(Y-Y_bar), axis=0)
# Function to compute KL Divergence for normal distributions
def kl_divergence(mu_x, sigma_x, mu_y, sigma_y):
    return ((mu_x - mu_y)**2) / (2 * sigma_y**2) + (sigma_x**2) / (2 * sigma_y**2) - 0.5 + np.log(sigma_y / sigma_x)


# Correct key for comparison
correct_key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]


t_bar = np.sum(trace_arrayx, axis=0)/len(trace_arrayx)
o_t = np.sqrt(np.sum((trace_arrayx - t_bar)**2, axis=0))

cparefs = [0] * 16 #put your key byte guess correlations here
bestguess = [0] * 16 #put your key byte guesses here
startt= time.time()
for bnum in trange(0, 16):
    maxcpa = [0] * 256
    for kguess in range(0, 256):
        hws = np.array([[HW[aes_internal(textin[bnum],kguess)] for textin in textin_array]]).transpose()
        hws_bar = mean(hws)
        o_hws = std_dev(hws, hws_bar)
        correlation = cov(trace_arrayx, t_bar, hws, hws_bar)
        cpaoutput = correlation/(o_t*o_hws)
        maxcpa[kguess] = max(abs(cpaoutput))
    bestguess[bnum] = np.argmax(maxcpa)
    cparefs[bnum] = max(maxcpa)
print("Best Key Guess: ", end="")
for b in bestguess: print("%02x " % b, end="")
print("\n", cparefs)
endd= time.time()
elapsed_time = endd - startt
print(f"Elapsed time: {elapsed_time:.5f} seconds")
if bestguess == correct_key:
        print("Correct key found!")
else:
    differences = [i for i, (a, b) in enumerate(zip(bestguess, correct_key)) if a != b]
    print(f"Correct key not found! Differences at indices: {differences}")
    for i in differences:
        print(f"Index {i}: bestguess = {bestguess[i]}, correct_key = {correct_key[i]}")

## Find Leakage Time interval, and Leakage model

In [ ]:
from scipy.stats import pearsonr
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from Crypto.Cipher import AES  # PyCryptodome library for AES verification
import time


sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

def aes_internal(inputdata, key):
    return sbox[inputdata ^ key]

HW = [bin(n).count("1") for n in range(0, 256)]


def mean(X):
    return np.sum(X, axis=0)/len(X)

def std_dev(X, X_bar):
    return np.sqrt(np.sum((X-X_bar)**2, axis=0))

def cov(X, X_bar, Y, Y_bar):
    return np.sum((X-X_bar)*(Y-Y_bar), axis=0)
# Function to compute KL Divergence for normal distributions
def kl_divergence(mu_x, sigma_x, mu_y, sigma_y):
    return ((mu_x - mu_y)**2) / (2 * sigma_y**2) + (sigma_x**2) / (2 * sigma_y**2) - 0.5 + np.log(sigma_y / sigma_x)


# Correct key for comparison
correct_key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]


cparefs = [0] * 16 #put your key byte guess correlations here
bestguess = [0] * 16 #put your key byte guesses here
selected_byte = 0 # Define a byte
threshold = 0.105  # Define a threshold
# Example Usage
key_guess = correct_key[selected_byte]  # Example key guess
print(f"correct key is: {key_guess}, and byte is {selected_byte}")
textin_array_np = np.array(textin_array)
plaintexts = textin_array_np[:, int(selected_byte)]


def extract_leakage_model(textin_array, key_guess):
    leakage_model = []
    for pt in textin_array:
        # Compute input to S-box
        sbox_input = pt ^ key_guess

        # S-box output
        sbox_output = sbox[sbox_input]

        # Extract the first bit (MSB)
        first_bit = (sbox_output >> 7) & 1

        # Convert to +1/-1
        leakage_model.append(1 if first_bit == 1 else -1)
    return np.array(leakage_model)

# Compute correlation over time
def compute_correlation(trace_array, leakage_model):
    correlation_over_time = []
    for t in range(trace_array.shape[1]):  # Loop over time samples
        trace_at_time = trace_array[:, t]
        corr, _ = pearsonr(trace_at_time, leakage_model)
        correlation_over_time.append(abs(corr))  # Absolute correlation
    return np.array(correlation_over_time)

# Find significant leakage time intervals
def find_leakage_time_interval(correlation_over_time, threshold=0.1):
    return np.where(correlation_over_time > threshold)[0]


# Extract leakage model
leakage_model = extract_leakage_model(plaintexts, key_guess)

# Compute correlation over time
##correlation_over_time = compute_correlation(trace_arrayx, leakage_model)

# Plot correlation over time
##plt.plot(correlation_over_time)
##plt.title("Correlation Over Time for First Bit Leakage Model")
##plt.xlabel("Time Samples")
##plt.ylabel("Correlation")
##plt.show()

# Identify leakage time interval

##leakage_time_indices = find_leakage_time_interval(correlation_over_time, threshold=threshold)
##print(f"Leakage Time Indices: {leakage_time_indices}")

In [ ]:
## Correct Version
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from Crypto.Cipher import AES  # PyCryptodome library for AES verification
import time


sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

def aes_internal(inputdata, key):
    return sbox[inputdata ^ key]

HW = [bin(n).count("1") for n in range(0, 256)]


def mean(X):
    return np.sum(X, axis=0)/len(X)

def std_dev(X, X_bar):
    return np.sqrt(np.sum((X-X_bar)**2, axis=0))

def cov(X, X_bar, Y, Y_bar):
    return np.sum((X-X_bar)*(Y-Y_bar), axis=0)
# Function to compute KL Divergence for normal distributions
def kl_divergence(mu_x, sigma_x, mu_y, sigma_y):
    return ((mu_x - mu_y)**2) / (2 * sigma_y**2) + (sigma_x**2) / (2 * sigma_y**2) - 0.5 + np.log(sigma_y / sigma_x)


# Correct key for comparison
correct_key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]


t_bar = np.sum(trace_arrayx, axis=0)/len(trace_arrayx)
o_t = np.sqrt(np.sum((trace_arrayx - t_bar)**2, axis=0))

cparefs = [0] * 16 #put your key byte guess correlations here
bestguess = [0] * 16 #put your key byte guesses here
startt= time.time()
for bnum in trange(0, 16):
    maxcpa = [0] * 256
    for kguess in range(0, 256):
        hws = np.array([[HW[aes_internal(textin[bnum],kguess)] for textin in textin_array]]).transpose()
        hws_bar = mean(hws)
        o_hws = std_dev(hws, hws_bar)
        correlation = cov(trace_arrayx, t_bar, hws, hws_bar)
        cpaoutput = correlation/(o_t*o_hws)
        maxcpa[kguess] = max(abs(cpaoutput))
    bestguess[bnum] = np.argmax(maxcpa)
    cparefs[bnum] = max(maxcpa)
print("Best Key Guess: ", end="")
for b in bestguess: print("%02x " % b, end="")
print("\n", cparefs)
endd= time.time()
elapsed_time = endd - startt
print(f"Elapsed time: {elapsed_time:.5f} seconds")
if bestguess == correct_key:
        print("Correct key found!")
else:
    differences = [i for i, (a, b) in enumerate(zip(bestguess, correct_key)) if a != b]
    print(f"Correct key not found! Differences at indices: {differences}")
    for i in differences:
        print(f"Index {i}: bestguess = {bestguess[i]}, correct_key = {correct_key[i]}")


In [ ]:
## Correct Version


import numpy as np

# AES S-box (as provided in your snippet)
from scipy.stats import pearsonr
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from Crypto.Cipher import AES  # PyCryptodome library for AES verification
import time
import numpy as np
from scipy.stats import pearsonr, norm


sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

def aes_internal(inputdata, key):
    return sbox[inputdata ^ key]

HW = [bin(n).count("1") for n in range(0, 256)]


def mean(X):
    return np.sum(X, axis=0)/len(X)

def std_dev(X, X_bar):
    return np.sqrt(np.sum((X-X_bar)**2, axis=0))

def cov(X, X_bar, Y, Y_bar):
    return np.sum((X-X_bar)*(Y-Y_bar), axis=0)
# Function to compute KL Divergence for normal distributions
def kl_divergence(mu_x, sigma_x, mu_y, sigma_y):
    return ((mu_x - mu_y)**2) / (2 * sigma_y**2) + (sigma_x**2) / (2 * sigma_y**2) - 0.5 + np.log(sigma_y / sigma_x)


def pearson_confidence_interval(x, y, alpha=0.05):
    # Calculate Pearson's correlation coefficient using scipy.stats.pearsonr
    r, p_value = pearsonr(x, y)
    n = len(x)

    # Fisher's z-transform to approximate normality
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)

    # Calculate the critical z-value
    z_critical = norm.ppf(1 - alpha / 2)

    # Compute confidence interval in z-space
    z_interval = z + np.array([-1, 1]) * z_critical * se

    # Transform back to correlation scale
    r_interval = np.tanh(z_interval)

    return r, r_interval, p_value

###############################################################################
# Single-bit HD model extraction
###############################################################################
def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=7):
    """
    Compute the single-bit HD between the input to the first-round S-box
    and its output for a chosen bit.

    :param plaintexts: 1D array/list of plaintext bytes (for selected byte index).
    :param key_byte:    The correct key byte for that position.
    :param selected_bit: Bit index to check (7 = MSB, 0 = LSB).
    :return:           1D numpy array with HD values (-1 or 1) for each plaintext.
    """
    leakage_model = []
    for pt in plaintexts:
        # 1) Previous value: input to the first S-box (pt XOR key)
        sbox_in = pt ^ key_byte
        prev_bit = (sbox_in >> selected_bit) & 0x1

        # 2) Current value: S-box output
        sbox_out = sbox[sbox_in]
        curr_bit = (sbox_out >> selected_bit) & 0x1

        # 3) Single-bit Hamming Distance = XOR of the bits
        hd_bit = prev_bit ^ curr_bit
        leakage_model.append(1 if hd_bit == 1 else -1)

    return np.array(leakage_model)
# Compute correlation over time
def compute_correlation(trace_array, leakage_model):
    correlation_over_time = []
    for t in range(trace_array.shape[1]):  # Loop over time samples
        trace_at_time = trace_array[:, t]
        corr, _ = pearsonr(trace_at_time, leakage_model)
        correlation_over_time.append(corr)
    return np.array(correlation_over_time)

# Find significant leakage time intervals
def find_leakage_time_interval(correlation_over_time, threshold=0.1):
    return np.where(abs(correlation_over_time) > threshold)[0]

###############################################################################
# Example usage
###############################################################################
if __name__ == "__main__":
    tistart= time.time()
    selected_byte = 0 # Define a byte
    chosen_bit = 0  # or any bit in [0..7]
    threshold = 0.105#0.110  # Define a threshold
    corkey = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6, 0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]
    key_guess = corkey[selected_byte]
    NTR=600
    print(f"correct key is: {key_guess}, and byte is {selected_byte}")
    textin_array_np = np.array(textin_array)
    plaintexts = textin_array_np[:, int(selected_byte)]
    # Generate random plaintext bytes for demonstration
    # Choose which bit to analyze (7 = MSB, 0 = LSB)


    # Extract single-bit HD leakage model
    hd_model = extract_leakage_model_hd_bit(
        plaintexts,
        key_guess,
        selected_bit=chosen_bit
    )
    print(f"HD model is: {len(hd_model)}")
    # Now, 'hd_model' is a 1D array of length N containing {0,1}
    #  - 0 means no bit flip from S-box input to output for that bit
    #  - 1 means the bit flipped from S-box input to output
    print("HD Model for Single Bit (0 or 1) =", hd_model)
    # Compute correlation over time
    correlation_over_time = compute_correlation(trace_arrayx, hd_model[0:NTR])
    print(f"timr is: {time.time()-tistart}")
    plt.plot(correlation_over_time)
    plt.title("Correlation Over Time for First Bit Leakage Model")
    plt.xlabel("Time Samples")
    plt.ylabel("Correlation")
    plt.show()
    # Identify leakage time interval
    num_time_steps = trace_arrayx.shape[1]

# Initialize an array to hold the correlation values
    correlations = np.zeros(num_time_steps)

# Calculate correlation at each time step
    for t in range(num_time_steps):
    # Extract the power measurements at time t (shape: (422,))
        P_t = trace_arrayx[:, t]

    # --- Method 1: Using np.corrcoef ----------------------------------
        #correlations[t] = np.corrcoef(L, P_t)[0, 1]

    # --- (Alternative) Method 2: Using the definition of covariance ---
        cov_L_Pt = np.mean((hd_model[0:NTR] - np.mean(hd_model[0:NTR])) * (P_t - np.mean(P_t)))
        std_L = np.std(hd_model[0:NTR], ddof=1)
        std_Pt = np.std(P_t, ddof=1)
        correlations[t] = cov_L_Pt / (std_L * std_Pt)

# Plot the correlation values
    plt.plot(correlations)
    plt.xlabel('Time Index')
    plt.ylabel('Correlation')
    plt.title('Correlation between Leakage Model and Power Traces over Time')
    plt.grid(True)
    plt.show()
    leakage_time_indices = find_leakage_time_interval(correlation_over_time, threshold=threshold)
    print(f"Leakage Time Indices: {leakage_time_indices}")


In [ ]:
leakage_model = hd_model

print(leakage_model)

### Leakage time calculate

In [ ]:
import numpy as np
import os

# Directory containing the data files
data_directory = path

# Initialize a list to store all the numpy arrays
data_arraystime = []

# Iterate through all 422 files

file_name = f"Result/plot_data_check_1.data"
file_path = os.path.join(data_directory, file_name)

    # Initialize a list to store the second values for the current file
first_values = []

    # Read the file
with open(file_path, "r") as file:
    for line in file:
            # Ignore comment lines starting with '#'
        if line.startswith("#"):
            continue

            # Split the line into two values
        parts = line.strip().split()
        if len(parts) == 2:
                # Extract the second value
            first_values.append(float(parts[0]))

    # Convert the list of second values to a NumPy array and store it
data_arraystime.append(np.array(first_values))

# Now `data_arrays` contains 422 NumPy arrays, each corresponding to one file.
# Example: Print the shape of the first array to verify
print(f"First array shape: {np.shape(data_arraystime)}")
print(f"First array shape: {data_arraystime[0].shape}")

time_start = data_arraystime[0][int(min(leakage_time_indices))]  # Start time in ns
time_end = data_arraystime[0][int(max(leakage_time_indices))]  # End time in ns

print(f"time idx = {int(min(leakage_time_indices))}, time_start: {time_start}")
print(f"time idx = {int(max(leakage_time_indices))}, time_end: {time_end}")


In [ ]:
import os
import re
import warnings
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import matplotlib.pyplot as plt  # for plotting (optional)

###############################################################################
# 1) Single-file parser
###############################################################################
def parse_vcd_signals_per_clock(vcd_file, clock_signal_name):
    """
    Parse a single VCD file, capturing integer values of all signals
    at each rising edge of the given clock signal.
    Returns a dict: {signal_name: [val_per_clock, ...], ...}
    """
    signal_map       = {}
    signal_values    = {}
    signal_waveforms = {}
    clock_id         = None

    in_dumpvars  = False
    current_time = 0

    with open(vcd_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # -----------------------------------------------------------------
            # Capture VCD $var definitions to build a map from ID -> signal_name
            # -----------------------------------------------------------------
            if line.startswith("$var"):
                parts = line.split()
                if len(parts) >= 5:
                    sig_id = parts[3]
                    name_tokens = parts[4:-1]
                    sig_name = " ".join(name_tokens).strip()

                    signal_map[sig_id] = sig_name
                    signal_values[sig_id] = 'x'
                    if sig_name == clock_signal_name:
                        clock_id = sig_id

                    signal_waveforms[sig_name] = []
                continue

            if line.startswith("$dumpvars"):
                in_dumpvars = True
                continue
            if line.startswith("$end") and in_dumpvars:
                in_dumpvars = False
                continue

            # -----------------------------------------------------------------
            # Time markers: #0, #10, #100, etc.
            # -----------------------------------------------------------------
            if line.startswith('#'):
                try:
                    current_time = int(line[1:])
                except ValueError:
                    current_time = 0
                continue

            # -----------------------------------------------------------------
            # Value changes: single-bit or multi-bit lines
            # -----------------------------------------------------------------
            if in_dumpvars or (current_time >= 0):
                # Single-bit: "0A", "1A", "xA"
                m_single = re.match(r'([01xXzZ])(\S+)', line)
                if m_single:
                    val, sid = m_single.groups()
                    if sid in signal_values:
                        val = val.lower()
                        val = '0' if (val == 'x' or val == 'z') else val
                        signal_values[sid] = val

                        # If this is the clock toggling, record a new "sample"
                        if sid == clock_id:
                            # For every known signal, push current val into waveform
                            for s_id, curr_val in signal_values.items():
                                s_name = signal_map[s_id]
                                int_val = 1 if curr_val == '1' else 0
                                signal_waveforms[s_name].append(int_val)
                else:
                    # Multi-bit: "b1010 A"
                    m_bus = re.match(r'b([01xXzZ]+)\s+(\S+)', line)
                    if m_bus:
                        bits_str, sid = m_bus.groups()
                        if sid in signal_values:
                            bits_str = bits_str.lower().replace('x','0').replace('z','0')
                            signal_values[sid] = bits_str

                            # If this is the clock toggling, record a new "sample"
                            if sid == clock_id:
                                for s_id, curr_val in signal_values.items():
                                    s_name = signal_map[s_id]
                                    curr_val = curr_val.lower().replace('x','0').replace('z','0')
                                    int_val = int(curr_val, 2)
                                    signal_waveforms[s_name].append(int_val)

    return signal_waveforms

###############################################################################
# 2) Parallel parsing of multiple VCDs
###############################################################################
def parse_vcd_file_wrapper(vcd_file, clock_signal):
    """
    Helper for parallel usage. Calls parse_vcd_signals_per_clock on one file.
    """
    return parse_vcd_signals_per_clock(vcd_file, clock_signal)

def parse_all_vcds_in_parallel(folder, clock_signal, total_files=600):
    """
    Spawn parallel tasks to parse each vcdN.vcd.
    Returns a list of dictionaries, one per file (in order).
    """
    filepaths = [os.path.join(folder, f"vcd{i}.vcd") for i in range(1, total_files + 1)]
    all_results = [None]*total_files

    with ProcessPoolExecutor() as executor:
        futures_map = {}
        for i, path in enumerate(filepaths):
            fut = executor.submit(parse_vcd_file_wrapper, path, clock_signal)
            futures_map[fut] = i

        for fut in tqdm(as_completed(futures_map), total=total_files, desc="Parsing VCDs"):
            i = futures_map[fut]
            try:
                result = fut.result()
                all_results[i] = result
            except Exception as e:
                print(f"Error parsing {filepaths[i]}: {e}")
                all_results[i] = {}

    return all_results

###############################################################################
# 3) Unify signals + build arrays
###############################################################################
def unify_signals_and_build_arrays(all_parsed):
    """
    all_parsed: list of dicts, each dict: { signal_name: [value0, value1, ...] }
    We unify across signals so that signals_dict[sig_name] is a 2D NumPy array
      shaped (num_traces, max_len), containing integer waveforms for that signal
      for each trace (VCD file).
    """
    num_vcds = len(all_parsed)
    all_signals = set()
    for d in all_parsed:
        all_signals.update(d.keys())
    all_signals = sorted(list(all_signals))  # stable ordering

    signals_dict = {}
    for sig_name in tqdm(all_signals, desc="Unifying signals"):
        waveforms = []
        lengths   = []
        for i in range(num_vcds):
            wave_i = all_parsed[i].get(sig_name, [])
            waveforms.append(wave_i)
            lengths.append(len(wave_i))

        max_len = max(lengths)
        arr = np.zeros((num_vcds, max_len), dtype=np.int32)

        for i in range(num_vcds):
            wave_i = waveforms[i]
            if len(wave_i) < max_len:
                if len(wave_i) > 0:
                    arr[i, :len(wave_i)] = wave_i
                if len(wave_i) > 0 and len(wave_i) < max_len:
                    print(f"Warning: file #{i+1}, signal '{sig_name}' has "
                          f"{len(wave_i)} edges < {max_len}. Padded with 0.")
            else:
                arr[i, :] = wave_i

        signals_dict[sig_name] = arr

    return signals_dict

###############################################################################
# 4) Toggle array builder
###############################################################################
def build_toggle_array(signal_array):
    """
    signal_array: shape (num_traces, num_clocks)

    Return toggles: shape (num_traces, num_clocks), where
      toggles[i, t] = +1 if signal_array[i, t] != signal_array[i, t-1], else -1
    The first column toggles[i, 0] = -1 by convention (no prior sample).
    """
    num_traces, num_clocks = signal_array.shape
    toggles = np.full((num_traces, num_clocks), -1, dtype=np.int8)

    for t in range(1, num_clocks):
        toggles[:, t] = np.where(signal_array[:, t] != signal_array[:, t-1], +1, -1)

    return toggles

###############################################################################
# 5) Dot product function (per clock sample)
###############################################################################
def compute_dot_product_per_time(toggle_array, leakage_model):
    """
    toggle_array:  shape (num_traces, num_clocks)
    leakage_model: shape (num_traces,)
    Returns an array of shape (num_clocks,), where
      out[t] = dot( toggle_array[:,t], leakage_model )
    """
    num_traces, num_clocks = toggle_array.shape
    dot_products = np.zeros(num_clocks, dtype=float)

    for t in range(num_clocks):
        column = toggle_array[:, t]  # shape: (num_traces,)
        dot_products[t] = np.dot(column, leakage_model)

    return dot_products

###############################################################################
# 6) Example S-box + function to build a "bit-level HD model"
###############################################################################
sbox = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]

def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=0):
    """
    Simple example: for each plaintext, compute the bit-level Hamming Distance
    between the input bit and output bit of S-box.
    Return an array of +1 or -1.
    """
    leakage_model = []
    for pt in plaintexts:
        sbox_in  = pt ^ key_byte
        bit_in   = (sbox_in >> selected_bit) & 0x1
        sbox_out = sbox[sbox_in]
        bit_out  = (sbox_out >> selected_bit) & 0x1
        hd_bit   = bit_in ^ bit_out
        leakage_model.append(+1 if hd_bit == 1 else -1)
    return np.array(leakage_model, dtype=np.float64)

###############################################################################
# 7) Main usage
###############################################################################
if __name__ == "__main__":
    # ------------------ (A) Setup ------------------
    vcd_folder = "/media/abish/Extreme Pro/Sbox_600/vcdssbox"#vcdssbox"
    clock_name = "SYSCLK_P"
    total_vcds = 600

    # Example: /path/to/plaintexts.txt
    txt_path = "/media/abish/Extreme Pro/Sbox_600/plaintexts.txt"
    # The plaintexts file is assumed to have lines like:
    #   #0 : 00 11 22 33 ...
    #   #1 : aa bb cc dd ...
    # so we parse after the colon for hex values.

    # ------------------ (B) Load plaintexts ------------------
    hex_arrays = []
    with open(txt_path, "r") as file:
        for line in file:
            parts = line.strip().split(":")
            if len(parts) == 2:
                hex_values = parts[1].strip().split()
                hex_arrays.append(hex_values)

    # Convert to 2D array (num_files, bytes_per_line)
    textin_array = []
    for row in hex_arrays:
        textin_array.append([int(x, 16) for x in row])
    textin_array = np.array(textin_array, dtype=np.uint8)
    print("textin_array shape:", textin_array.shape)

    # Example AES key
    corkey = [0x2b,0x7e,0x15,0x16,0x28,0xae,0xd2,0xa6,
              0xab,0xf7,0x15,0x88,0x09,0xcf,0x4f,0x3c]
    selected_byte = 0
    chosen_bit    = 0  # LSB
    key_guess     = corkey[selected_byte]

    # Build the leakage model (shape: (num_traces,))
    plaintext_for_each_trace = textin_array[:, selected_byte]
    leakage_model = extract_leakage_model_hd_bit(plaintext_for_each_trace, key_guess, selected_bit=chosen_bit)
    print(f"Built HD model for {total_vcds} traces (bit={chosen_bit}).")

    # ------------------ (C) Parse VCDs ------------------
    print(f"Parsing {total_vcds} VCD files in parallel ...")
    all_parsed = parse_all_vcds_in_parallel(vcd_folder, clock_signal=clock_name, total_files=total_vcds)
    print("Done parsing.\n")

    # ------------------ (D) Unify signals into arrays ------------------
    print("Unifying signals across all VCDs and building arrays...")
    signals_dict = unify_signals_and_build_arrays(all_parsed)
    print(f"Done. Found {len(signals_dict)} unique signals.\n")

    # ------------------ (E) For each signal, build toggles & compute dot product ------------------
    toggle_results = []
    for sig_name, signal_array in tqdm(signals_dict.items(), desc="Toggle dot products"):
        # 1) Build toggles: shape (num_traces, num_clocks)
        toggle_array = build_toggle_array(signal_array)

        # 2) Compute dot product vs. leakage model for each clock
        dp_over_time = compute_dot_product_per_time(toggle_array, leakage_model)

        # 3) Find max absolute dot product and its clock index

        best_idx = np.argmax(np.abs(dp_over_time))
        max_val  = dp_over_time[best_idx]
        toggle_results.append((sig_name, dp_over_time, max_val, best_idx))

    # Sort by max absolute dot product
    toggle_results.sort(key=lambda x: x[2], reverse=True)

    # ------------------ (F) Print top 20 signals ------------------
    print("\n=== Top Toggle-Dot-Product Signals ===")
    for i, (sig_name, dp_arr, max_val, best_idx) in enumerate(toggle_results[:20]):
        print(f"{i+1:2d}) {sig_name}: max abs dot product={max_val:.4f} at clock={best_idx}")

    # ------------------ (G) Optional: Plot some signals ------------------
    # Example: plot top 1
    # if toggle_results:
    #     top_sig, top_dp, _, _ = toggle_results[0]
    #     plt.figure()
    #     plt.plot(top_dp)
    #     plt.title(f"Dot Product Over Time for {top_sig}")
    #     plt.xlabel("Clock sample")
    #     plt.ylabel("Dot product")
    #     plt.show()


In [ ]:
import os
import re
import time
import warnings
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import matplotlib.pyplot as plt

###############################################################################
# 1) Single VCD parser
###############################################################################
def parse_vcd_signals_per_clock(vcd_file, clock_signal_name):
    """
    Parse a single VCD file, capturing integer values of all signals
    at each rising edge of 'clock_signal_name'.
    Returns a dict: { signal_name: [val_per_clock, ...], ... }
    """
    signal_map       = {}
    signal_values    = {}
    signal_waveforms = {}
    clock_id         = None

    in_dumpvars  = False
    current_time = 0

    with open(vcd_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Capture $var definitions => ID -> signal_name
            if line.startswith("$var"):
                parts = line.split()
                if len(parts) >= 5:
                    sig_id   = parts[3]
                    name_tok = parts[4:-1]
                    sig_name = " ".join(name_tok).strip()

                    signal_map[sig_id]       = sig_name
                    signal_values[sig_id]    = 'x'
                    signal_waveforms[sig_name] = []

                    if sig_name == clock_signal_name:
                        clock_id = sig_id
                continue

            if line.startswith("$dumpvars"):
                in_dumpvars = True
                continue
            if line.startswith("$end") and in_dumpvars:
                in_dumpvars = False
                continue

            # Time markers (#0, #100, etc.)
            if line.startswith('#'):
                try:
                    current_time = int(line[1:])
                except ValueError:
                    current_time = 0
                continue

            # Value changes
            if in_dumpvars or (current_time >= 0):
                # Single-bit: "0A", "1A", "xA"
                m_single = re.match(r'([01xXzZ])(\S+)', line)
                if m_single:
                    val, sid = m_single.groups()
                    if sid in signal_values:
                        val = val.lower()
                        val = '0' if val in ('x', 'z') else val
                        signal_values[sid] = val

                        # If the clock just toggled, record new sample for all signals
                        if sid == clock_id:
                            for s_id, curr_val in signal_values.items():
                                s_name = signal_map[s_id]
                                int_val = 1 if curr_val == '1' else 0
                                signal_waveforms[s_name].append(int_val)

                else:
                    # Multi-bit: "b1010 A"
                    m_bus = re.match(r'b([01xXzZ]+)\s+(\S+)', line)
                    if m_bus:
                        bits_str, sid = m_bus.groups()
                        if sid in signal_values:
                            bits_str = bits_str.lower().replace('x','0').replace('z','0')
                            signal_values[sid] = bits_str

                            if sid == clock_id:
                                for s_id, curr_val in signal_values.items():
                                    s_name   = signal_map[s_id]
                                    curr_val = curr_val.lower().replace('x','0').replace('z','0')
                                    int_val  = int(curr_val, 2)
                                    signal_waveforms[s_name].append(int_val)

    return signal_waveforms


###############################################################################
# 2) Parallel parse for all VCD files
###############################################################################
def parse_vcd_file_wrapper(args):
    """
    Helper for parallel usage. We pass a tuple of (vcd_file, clock_signal).
    """
    vcd_file, clock_signal = args
    return parse_vcd_signals_per_clock(vcd_file, clock_signal)


def parse_all_vcds_in_parallel(folder, clock_signal, total_files=600):
    """
    Spin up parallel tasks to parse each file vcdN.vcd.
    Returns a list of dictionaries, one per file index.
    """
    filepaths = [os.path.join(folder, f"vcd{i}.vcd") for i in range(1, total_files + 1)]
    all_results = [None]*total_files

    tasks = [(filepaths[i], clock_signal) for i in range(total_files)]

    with ProcessPoolExecutor() as executor:
        futures_map = {executor.submit(parse_vcd_file_wrapper, t): idx for idx, t in enumerate(tasks)}

        for fut in tqdm(as_completed(futures_map), total=total_files, desc="Parsing VCDs"):
            i = futures_map[fut]
            try:
                res = fut.result()
                all_results[i] = res
            except Exception as e:
                print(f"[!] Error parsing {filepaths[i]}: {e}")
                all_results[i] = {}

    return all_results


###############################################################################
# 3) Unify signals
###############################################################################
def unify_signals_and_build_arrays(all_parsed):
    """
    Given a list of dicts (one per VCD), unify so that signals_dict[sig_name]
    is a 2D np.array of shape (num_traces, max_len).
    """
    num_vcds = len(all_parsed)
    all_signals = set()
    for d in all_parsed:
        all_signals.update(d.keys())
    all_signals = sorted(list(all_signals))

    signals_dict = {}
    for sig_name in tqdm(all_signals, desc="Unifying signals"):
        waveforms = []
        lengths = []
        for i in range(num_vcds):
            wave_i = all_parsed[i].get(sig_name, [])
            waveforms.append(wave_i)
            lengths.append(len(wave_i))

        max_len = max(lengths)
        arr = np.zeros((num_vcds, max_len), dtype=np.int32)

        for i in range(num_vcds):
            wave_i = waveforms[i]
            if len(wave_i) > 0:
                arr[i, :len(wave_i)] = wave_i
            if len(wave_i) < max_len and len(wave_i) > 0:
                print(f"[!] Warning: file {i+1}, signal '{sig_name}' has "
                      f"{len(wave_i)} edges < {max_len}. Padded with 0.")

        signals_dict[sig_name] = arr

    return signals_dict


###############################################################################
# 4) Build toggle array
###############################################################################
def build_toggle_array(signal_array):
    """
    Given signal_array shape (num_traces, num_clocks),
    produce toggles of the same shape: +1 if toggles from previous clock, else -1.
    toggles[:, 0] = -1 by convention.
    """
    num_traces, num_clocks = signal_array.shape
    toggles = np.full((num_traces, num_clocks), -1, dtype=np.int8)

    for t in range(1, num_clocks):
        toggles[:, t] = np.where(signal_array[:, t] != signal_array[:, t - 1], +1, -1)

    return toggles


###############################################################################
# 5) Dot product at each clock sample
###############################################################################
def compute_dot_product_per_time(toggle_array, leakage_model):
    """
    toggle_array: shape (num_traces, num_clocks)
    leakage_model: shape (num_traces,)
    Return dp[t] = sum(toggle_array[i, t] * leakage_model[i]) for i in [0..num_traces-1].
    """
    num_traces, num_clocks = toggle_array.shape
    dot_products = np.zeros(num_clocks, dtype=float)

    # vectorized approach:
    # dot_products = toggle_array.T @ leakage_model
    # but let's keep the loop for clarity:
    for t in range(num_clocks):
        column = toggle_array[:, t]
        dot_products[t] = np.dot(column, leakage_model)

    return dot_products


###############################################################################
# 6) Example: HD model for a single bit
###############################################################################
sbox = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]

def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=0):
    """
    For each plaintext, compute the HD between the input bit and output bit of S-box.
    Return array of +1 / -1.
    """
    model = []
    for pt in plaintexts:
        sbox_in  = pt ^ key_byte
        bit_in   = (sbox_in >> selected_bit) & 0x1
        sbox_out = sbox[sbox_in]
        bit_out  = (sbox_out >> selected_bit) & 0x1
        hd_bit   = bit_in ^ bit_out
        model.append(+1 if hd_bit == 1 else -1)
    return np.array(model, dtype=np.float64)


###############################################################################
# 7) Function to compute toggle dot-products for a given key guess
###############################################################################
def compute_toggle_dot_for_key_guess(key_guess, plaintexts, toggles_dict):
    """
    key_guess: int in [0..127] or [0..255], etc.
    plaintexts: shape (num_traces,)
    toggles_dict: { signal_name: 2D toggle array [num_traces, num_clocks] }

    Returns:
      ( key_guess,
        list of (sig_name, max_abs_dot, best_clock_idx),
        iteration_runtime_seconds )
    """
    start_time = time.time()

    # 1) Build the leakage model
    #   For demonstration, we fix selected_bit=0 (LSB)
    #   If you want multi-bit or multiple bytes, adapt accordingly
    selected_bit = 0
    leakage_model = extract_leakage_model_hd_bit(plaintexts, key_guess, selected_bit=selected_bit)

    # 2) For each signal, compute the max absolute dot across time
    results_for_signals = []
    for sig_name, toggle_array in toggles_dict.items():
        dp_over_time = compute_dot_product_per_time(toggle_array, leakage_model)
        max_val  = np.max(np.abs(dp_over_time))
        best_idx = np.argmax(np.abs(dp_over_time))
        results_for_signals.append((sig_name, max_val, best_idx))

    # You might only want to store the top 1 or top N signals to save memory.
    # For demonstration, let's keep them all. If large, consider truncating.

    end_time = time.time()
    iteration_time = end_time - start_time

    return (key_guess, results_for_signals, iteration_time)


###############################################################################
# 8) Main
###############################################################################
if __name__ == "__main__":
    # ----------------- (A) Paths and Setup -----------------
    vcd_folder = "/media/abish/Extreme Pro/Sbox_600/vcdssbox"#vcdssbox"
    clock_name = "SYSCLK_P"
    total_vcds = 600

    # Example: /path/to/plaintexts.txt
    txt_path = "/media/abish/Extreme Pro/Sbox_600/plaintexts.txt"
    hex_arrays = []
    with open(txt_path, "r") as file:
        for line in file:
            parts = line.strip().split(":")
            if len(parts) == 2:
                hex_values = parts[1].strip().split()
                hex_arrays.append(hex_values)

    # Convert to 2D array (num_vcds, bytes_per_line)
    textin_array = []
    for row in hex_arrays:
        textin_array.append([int(x, 16) for x in row])
    textin_array = np.array(textin_array, dtype=np.uint8)
    print("textin_array shape:", textin_array.shape)

    # In this example, let's only brute-force key guesses in [0..127].
    # For full AES keyspace of one byte, do range(256).
    possible_key_guesses = range(128)

    # We'll use only one AES byte (e.g., byte 0).
    selected_byte_index = 0
    # plaintext_for_each_trace => shape (600,) if we have 600 VCDs
    plaintext_for_each_trace = textin_array[:, selected_byte_index]

    # ----------------- (B) Parse VCDs in parallel -----------------
    parse_start = time.time()
    all_parsed = parse_all_vcds_in_parallel(vcd_folder, clock_signal=clock_name, total_files=total_vcds)
    parse_end = time.time()
    print(f"[Timing] Parsing all VCDs took {parse_end - parse_start:.2f} s.")

    # ----------------- (C) Unify signals -----------------
    unify_start = time.time()
    signals_dict = unify_signals_and_build_arrays(all_parsed)
    unify_end = time.time()
    print(f"[Timing] Unifying signals took {unify_end - unify_start:.2f} s.")
    print(f"Found {len(signals_dict)} unique signals.")

    # ----------------- (D) Pre-build toggles for each signal -----------------
    #  This does NOT depend on the key guess => we do it once.
    toggle_start = time.time()
    toggles_dict = {}
    for sig_name, sig_array in tqdm(signals_dict.items(), desc="Build toggles"):
        toggles_dict[sig_name] = build_toggle_array(sig_array)
    toggle_end = time.time()
    print(f"[Timing] Building toggles took {toggle_end - toggle_start:.2f} s.")

    # ----------------- (E) For each key guess, compute dot-product in parallel -----------------
    def parallel_key_guess_wrapper(k):
        # We pass the minimal data needed: the single plaintext column + toggles
        return compute_toggle_dot_for_key_guess(k, plaintext_for_each_trace, toggles_dict)

    all_keys_start = time.time()
    iteration_times = []

    # Option A) PARALLEL approach across key guesses
    with ProcessPoolExecutor() as executor:
        futures = [executor.submit(parallel_key_guess_wrapper, k) for k in possible_key_guesses]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Key guesses"):
            try:
                key_guess, results_for_signals, iter_time = fut.result()
                iteration_times.append(iter_time)
                # Optionally store or print partial results:
                # e.g. find top signals for this key guess
                # sorted_signals = sorted(results_for_signals, key=lambda x: x[1], reverse=True)
                # top_signal_name, top_val, top_idx = sorted_signals[0]
                # print(f"Key 0x{key_guess:02X}: best signal={top_signal_name} val={top_val} clock={top_idx}")
            except Exception as e:
                print(f"[!] Error in key guess job: {e}")

    # Option B) SERIAL approach (if you prefer to measure iteration time easily):
    # all_results = []
    # for k in tqdm(possible_key_guesses, desc="Key guesses"):
    #     r = parallel_key_guess_wrapper(k)
    #     all_results.append(r)
    #     iteration_times.append(r[-1])

    all_keys_end = time.time()
    total_time_for_all_keys = all_keys_end - all_keys_start

    # ----------------- (F) Print timing statistics -----------------
    print("\n--- TIMING SUMMARY ---")
    print(f"VCD parsing time:   {parse_end - parse_start:.2f} s")
    print(f"Unify signals time: {unify_end - unify_start:.2f} s")
    print(f"Build toggles time: {toggle_end - toggle_start:.2f} s")

    print(f"\nKey guess computations: {len(possible_key_guesses)} total")
    print(f"Total time for all keys: {total_time_for_all_keys:.2f} s")

    if iteration_times:
        avg_iter = np.mean(iteration_times)
        min_iter = np.min(iteration_times)
        max_iter = np.max(iteration_times)
        print(f"  Per-iteration stats (in parallel):")
        print(f"    Average = {avg_iter:.4f} s")
        print(f"    Min     = {min_iter:.4f} s")
        print(f"    Max     = {max_iter:.4f} s")

    # -------------- Optional: do something with the final results --------------
    # In the parallel approach above, you might have stored partial results in a global or
    # appended them to a manager structure. Or you can switch to the serial approach if you
    # want a local all_results list to analyze further.


In [ ]:
toggle_results

In [ ]:
from tqdm import tqdm  # Add this at the top

def analyze_high_correlation_signals(ranked_signals, netlist_file, tech_file, power_report_file):
    gates_dict = parse_gate_definitions(tech_file)
    power_data = extract_power_per_instance(power_report_file)
    signal_gate_mapping = {}

    for (signal, dp_arr, correlation, best_idx)  in tqdm(ranked_signals, desc="Analyzing Signals", unit="sig"):
        nesi = False
        #if abs(correlation) <= analysis_correlation_limitation:
           # continue

        processed_signal = preprocess_signal_name(signal)
        connections = find_signal_in_netlist(processed_signal, netlist_file)

        signal_gate_mapping[signal] = {
            "correlation": correlation,
            "gates": [],
            "Impact": 0
        }

        instance_powers = []
        for connection in connections:
            gate_matchxc = re.search(r'(SC8T_\S+)\s+(\S+)\(', connection)
            if gate_matchxc is None:
                gate_matchxc = re.search(r'(SC8T_\S+)\s+(\S+)\s*\(', connection)

            if gate_matchxc:
                gate_name = gate_matchxc.group(1)
                instance_namef = gate_matchxc.group(2)
                instance_name = instance_namef.replace("\\", "")

                if gate_name in gates_dict:
                    gate_info = gates_dict[gate_name]
                    port_signal_map = parse_connection_ports(connection)
                    roles = []

                    for port, s_name in port_signal_map.items():
                        if s_name == processed_signal:
                            if port in gate_info["inputs"]:
                                roles.append("input")
                            if port in gate_info["outputs"]:
                                roles.append("output")

                    if roles:
                        signal_gate_mapping[signal]["gates"].append({
                            "gate_name": gate_name,
                            "gate_info": gate_info,
                            "instance_name": instance_name,
                            "connection": connection,
                            "roles": roles
                        })

                        if "output" in roles and not nesi:
                            for power_entry in power_data.keys():
                                power_instance_name = power_entry.split('/')[-1]
                                if instance_name == power_instance_name:
                                    instance_powers.append(power_data[power_entry])
                                    nesi = True
                                    break

        if instance_powers:
            average_power = sum(instance_powers) / len(instance_powers)
            signal_gate_mapping[signal]["Impact"] = average_power / total_Power

    return signal_gate_mapping


## Netlist process (get gates accourding to signal) + compute Impact factor
import re

analysis_correlation_limitation = 1
total_Power = 1.39427e-04


# Preprocess signal names
def preprocess_signal_name(signal):
    return signal.replace("][", "] [")


# Calculate the ranking based on Correlation x Average Power
def rank_signals_by_impact(signal_gate_mapping):
    ranked_signals = []

    for signal, data in signal_gate_mapping.items():
        # Skip signals with no calculated impact
        #if data["Impact"] > 0:
        # Handle NaN correlation by assuming it as 0
        try:
            correlation = data["correlation"]
            if correlation is None:
                correlation = 0  # Treat NaN as 0

                # Calculate the impact value
            impact_value = correlation * data["Impact"]

            # Find the gate instance name with the "output" role
            output_instance = None
            for gate in data["gates"]:
                if "output" in gate["roles"]:
                    output_instance = gate["instance_name"]
                    break

            # Add the signal, impact value, and output instance to the ranking list
            ranked_signals.append({
                "signal": signal,
                "impact_value": impact_value,
                "output_instance": output_instance,
                "correlation": correlation,  # Include correlation for debugging purposes
            })
        except KeyError as e:
            print(f"KeyError for signal '{signal}': Missing key {e}")
        except Exception as e:
            print(f"Unexpected error for signal '{signal}': {e}")
    # Sort signals by impact value in descending order
    ranked_signals.sort(key=lambda x: x["impact_value"], reverse=True)

    return ranked_signals


# Debugging Output: Verify Intermediate Data
def print_debug_data(signal_gate_mapping):
    print("\nSignal Data (Before Ranking):")
    for signal, data in signal_gate_mapping.items():
        print(f"Signal: {signal}, Correlation: {data['correlation']}, Impact: {data['Impact']:.6e}")
        for gate in data["gates"]:
            print(f"  Gate: {gate['gate_name']}, Instance: {gate['instance_name']}, Role: {gate['role']}")


# Debugging Output: Verify Ranked Signals
def print_ranked_signals(ranked_signals):
    print("\nRanked Signals by Impact Value:")
    for rank, entry in enumerate(ranked_signals, start=1):
        print(f"Rank {rank}: Signal = {entry['signal']}, "
              f"Impact Value = {entry['impact_value']:.6e}, "
              f"correlation = {entry['correlation']}, "
              f"Output Instance = {entry['output_instance']}")


def find_signal_in_netlist(signal, netlist_file):
    connections = []
    with open(netlist_file, 'r') as file:
        lines = file.readlines()

    # Create a regex pattern to match the exact signal name
    escaped_signal = re.escape(signal).replace(r'\[', r'\s*\[').replace(r'\]', r'\s*\]')
    signal_pattern = re.compile(rf'(?<!\w){escaped_signal}(?!\w)')  # Match the signal exactly

    for i, line in enumerate(lines):
        # Skip wire declarations
        if line.strip().startswith("wire"):
            continue

        # Check if the signal is in the current line
        if signal_pattern.search(line):  # Use regex search for exact match
            # Backtrack to find the start of the gate (starting with SC8T_)
            start_idx = i
            while start_idx > 0 and not lines[start_idx].strip().startswith("SC8T_"):
                start_idx -= 1

            # Forward track to find the end of the gate (ending with ;)
            end_idx = i
            while end_idx < len(lines) and ";" not in lines[end_idx]:
                end_idx += 1

            # Capture the full gate definition
            gate_definition = "".join(lines[start_idx:end_idx + 1]).strip()
            gate_definition_cleaned = re.sub(r'\s+', ' ', gate_definition)

            connections.append(gate_definition_cleaned)

    return connections


# Parse gate definitions from the technology file
def parse_gate_definitions(tech_file):
    gates_dict = {}
    module_def_regex = re.compile(r'module\s+(\S+)\s*\(([^)]*)\);')
    input_regex = re.compile(r'input\s+([^;]+);')
    output_regex = re.compile(r'output\s+([^;]+);')

    with open(tech_file, 'r') as vf:
        verilog_content = vf.read()

    for match in module_def_regex.finditer(verilog_content):
        gate_name, ports = match.groups()
        ports = [p.strip() for p in ports.split(',')]

        # Extract inputs and outputs
        inputs = []
        outputs = []
        start_idx = verilog_content.find(match.group(0)) + len(match.group(0))
        module_body = verilog_content[start_idx:verilog_content.find("endmodule", start_idx)]

        for input_match in input_regex.finditer(module_body):
            inputs.extend(input_match.group(1).split(','))

        for output_match in output_regex.finditer(module_body):
            outputs.extend(output_match.group(1).split(','))

        gates_dict[gate_name] = {
            "name": gate_name,
            "inputs": [i.strip() for i in inputs],
            "outputs": [o.strip() for o in outputs],
            "ports": ports
        }
    return gates_dict


# Parse port-to-signal mapping
def parse_connection_ports(connection):
    port_signal_regex = re.findall(r'\.(\w+)\s*\(\s*(.*?)\s*\)', connection)
    return {port: signal.replace("\\", "").strip() for port, signal in port_signal_regex}


# Extract power for each instance from the power report
def extract_power_per_instance(power_report_file):
    power_per_instance = {}
    with open(power_report_file, 'r') as file:
        for line in file:
            if re.match(r'^\s*[0-9eE.+-]+\s+[0-9eE.+-]+\s+[0-9eE.+-]+\s+[0-9eE.+-]+\s+/.+', line):
                parts = line.split()
                total_power = float(parts[-2])  # Extract total power
                instance_name = parts[-1]  # Extract instance name
                power_per_instance[instance_name] = total_power
    return power_per_instance



# Example Usage
if __name__ == "__main__":
    ranked_signals_new = []
    path = "/media/abish/Extreme Pro/sbox_600"
    netlist_file = path + "/" + "PROACT_topa_netlist.v"
    tech_file = path + "/" + "GF22FDX_SC8T_116CPP_BASE_DDC28UH.v"
    power_report_file = path + "/" + "Power_per_instance.txt"

    signal_gate_mapping = analyze_high_correlation_signals(toggle_results, netlist_file, tech_file,
                                                           power_report_file)

    # Print results
    ranked_signals_by_impact = rank_signals_by_impact(signal_gate_mapping)

    print_ranked_signals(ranked_signals_by_impact)

In [ ]:
import matplotlib.pyplot as plt

# Example: Extract impact values from ranked_signals_by_impact (replace this with your actual data)
impact_values = [entry["impact_value"] for entry in ranked_signals_by_impact]

# Sort impact values
impact_values = sorted(impact_values)

# Plotting the Histogram
plt.figure(figsize=(10, 6))
plt.hist(impact_values, bins=50, log=True, color='blue', edgecolor='black')
plt.xlabel('Leakage Impact Factor (LIF)', fontsize=12)
plt.ylabel('Number of Cells', fontsize=12)
plt.title('Leakage Impact Factor Distribution', fontsize=14)

# Highlight and Annotate Top Ranking Cells
top_3 = sorted(impact_values, reverse=True)[:3]  # Top 3 impact values
positions = [1, 2, 3]  # For ranking (1st, 2nd, 3rd)
for i, value in enumerate(top_3):
    plt.axvline(value, color='black', linestyle='--', linewidth=1)
    plt.text(value, 10, f"{positions[i]} Ranking Cell", rotation=90, fontsize=10, ha='right', va='bottom')

# Adjust the plot
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.savefig("NETLIF_distribution.pdf", bbox_inches="tight", transparent=True)
plt.savefig("NETLIF_correlation_distribution.svg", bbox_inches="tight", transparent=True)
plt.savefig("NETLIF_correlation_distribution.png", bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
import pandas as pd

# Define LIF ranges
lif_ranges = [
    (0.12, 0.2),
    (0.07, 0.12),
    (0.03, 0.07),
    (-0.2, 0.03),
]

# Initialize a dictionary to store the counts for each range
lif_distribution = {"LIF Range": [], "No. of Cells": []}

# Iterate through the defined ranges
for lif_min, lif_max in lif_ranges:
    # Count signals within the current range
    count = sum(
        lif_min <= entry["impact_value"] <= lif_max
        for entry in ranked_signals_by_impact
    )
    lif_distribution["LIF Range"].append(f"{lif_min} ~ {lif_max}")
    lif_distribution["No. of Cells"].append(count)

# Convert to DataFrame for a table-like representation
df_lif = pd.DataFrame(lif_distribution)

# Print the table
print("TABLE III: LIF Distribution Data")
print(df_lif.to_string(index=False))

# Optionally save the table to a CSV
df_lif.to_csv("table_III_LIF_distribution.csv", index=False)


In [ ]:
## New mthode test here

In [ ]:
import os
import re
import warnings
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.stats import pearsonr, ConstantInputWarning
from tqdm import tqdm
import matplotlib.pyplot as plt  # for plotting

###############################################################################
# 1) Single-file parser
###############################################################################
def parse_vcd_signals_per_clock(vcd_file, clock_signal_name):
    signal_map       = {}
    signal_values    = {}
    signal_waveforms = {}
    clock_id         = None

    in_dumpvars  = False
    current_time = 0

    with open(vcd_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith("$var"):
                parts = line.split()
                if len(parts) >= 5:
                    sig_id = parts[3]
                    name_tokens = parts[4:-1]
                    sig_name = " ".join(name_tokens).strip()

                    signal_map[sig_id] = sig_name
                    signal_values[sig_id] = 'x'
                    if sig_name == clock_signal_name:
                        clock_id = sig_id

                    signal_waveforms[sig_name] = []
                continue

            if line.startswith("$dumpvars"):
                in_dumpvars = True
                continue
            if line.startswith("$end") and in_dumpvars:
                in_dumpvars = False
                continue

            if line.startswith('#'):
                try:
                    current_time = int(line[1:])
                except ValueError:
                    current_time = 0
                continue

            if in_dumpvars or (current_time >= 0):
                # Single-bit: "0A", "1A", "xA" ...
                m_single = re.match(r'([01xXzZ])(\S+)', line)
                if m_single:
                    val, sid = m_single.groups()
                    if sid in signal_values:
                        val = val.lower()
                        val = '0' if (val == 'x' or val == 'z') else val
                        signal_values[sid] = val

                        if sid == clock_id:
                            for s_id, curr_val in signal_values.items():
                                s_name = signal_map[s_id]
                                int_val = 1 if curr_val == '1' else 0
                                signal_waveforms[s_name].append(int_val)
                else:
                    # Multi-bit: "b1010 A"
                    m_bus = re.match(r'b([01xXzZ]+)\s+(\S+)', line)
                    if m_bus:
                        bits_str, sid = m_bus.groups()
                        if sid in signal_values:
                            bits_str = bits_str.lower().replace('x','0').replace('z','0')
                            signal_values[sid] = bits_str

                            if sid == clock_id:
                                for s_id, curr_val in signal_values.items():
                                    s_name = signal_map[s_id]
                                    curr_val = curr_val.lower().replace('x','0').replace('z','0')
                                    int_val = int(curr_val, 2)
                                    signal_waveforms[s_name].append(int_val)

    return signal_waveforms


###############################################################################
# 2) Parallel wrapper
###############################################################################
def parse_vcd_file_wrapper(vcd_file, clock_signal):
    return parse_vcd_signals_per_clock(vcd_file, clock_signal)


def parse_all_vcds_in_parallel(folder, clock_signal, total_files=600):
    from concurrent.futures import ProcessPoolExecutor, as_completed
    filepaths = [os.path.join(folder, f"vcd{i}.vcd") for i in range(1, total_files + 1)]
    all_results = [None]*total_files

    with ProcessPoolExecutor() as executor:
        futures_map = {}
        for i, path in enumerate(filepaths):
            fut = executor.submit(parse_vcd_file_wrapper, path, clock_signal)
            futures_map[fut] = i

        for fut in tqdm(as_completed(futures_map), total=total_files, desc="Parsing VCDs"):
            i = futures_map[fut]
            try:
                result = fut.result()
                all_results[i] = result
            except Exception as e:
                print(f"Error parsing {filepaths[i]}: {e}")
                all_results[i] = {}

    return all_results


###############################################################################
# 3) Unify signals + build arrays
###############################################################################
def unify_signals_and_build_arrays(all_parsed):
    num_vcds = len(all_parsed)
    all_signals = set()
    for d in all_parsed:
        all_signals.update(d.keys())
    all_signals = sorted(list(all_signals))

    signals_dict = {}
    for sig_name in tqdm(all_signals, desc="Unifying signals"):
        waveforms = []
        lengths   = []
        for i in range(num_vcds):
            wave_i = all_parsed[i].get(sig_name, [])
            lengths.append(len(wave_i))
            waveforms.append(wave_i)

        max_len = max(lengths)
        arr = np.zeros((num_vcds, max_len), dtype=np.int32)
        for i in range(num_vcds):
            wave_i = waveforms[i]
            if len(wave_i) < max_len:
                if len(wave_i) > 0:
                    arr[i, :len(wave_i)] = wave_i
                if len(wave_i) > 0 and len(wave_i) < max_len:
                    print(f"Warning: file #{i+1}, signal '{sig_name}' has "
                          f"{len(wave_i)} edges < {max_len}. Padded with 0.")
            else:
                arr[i, :] = wave_i
        signals_dict[sig_name] = arr

    return signals_dict


###############################################################################
# 4) Correlation function
###############################################################################
def compute_correlationpppp(trace_array, leakage_model):
    num_traces, num_time = trace_array.shape
    correlation_over_time = np.zeros(num_time, dtype=float)

    for t in range(num_time):
        column = trace_array[:, t]
        if np.all(column == column[0]) or np.all(leakage_model == leakage_model[0]):
            correlation_over_time[t] = 0.0
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConstantInputWarning)
                r, _ = pearsonr(column, leakage_model)
                correlation_over_time[t] = r

    return correlation_over_time

def compute_correlation(trace_array, leakage_model):
    """
    Compute the raw dot product (column · leakage_model) for each time sample,
    instead of Pearson's correlation.
    """
    num_traces, num_time = trace_array.shape
    correlation_over_time = np.zeros(num_time, dtype=float)

    for t in range(num_time):
        column = trace_array[:, t]
        # Optional: Skip dot product if all values are constant
        if np.all(column == column[0]) or np.all(leakage_model == leakage_model[0]):
            correlation_over_time[t] = 0.0
        else:
            # Dot product
            correlation_over_time[t] = np.dot(column, leakage_model)

    return correlation_over_time

###############################################################################
# 4.5) Build toggle array
###############################################################################
def build_toggle_array(signal_array):
    """
    signal_array: shape (num_traces, num_clocks)

    Return toggles: shape (num_traces, num_clocks), where
      toggles[i, t] = +1 if signal_array[i, t] != signal_array[i, t-1] else -1
    The first column toggles[i, 0] is set to -1 by convention.
    """
    num_traces, num_clocks = signal_array.shape
    toggles = np.full((num_traces, num_clocks), -1, dtype=np.int8)

    for t in range(1, num_clocks):
        toggles[:, t] = np.where(signal_array[:, t] != signal_array[:, t-1], +1, -1)

    return toggles


###############################################################################
# Additional: Example s-box + HD model
###############################################################################
sbox = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]

def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=0):
    leakage_model = []
    for pt in plaintexts:
        sbox_in = pt ^ key_byte
        bit_in = (sbox_in >> selected_bit) & 0x1
        sbox_out = sbox[sbox_in]
        bit_out = (sbox_out >> selected_bit) & 0x1
        hd_bit = bit_in ^ bit_out
        leakage_model.append(1 if hd_bit == 1 else -1)
    return np.array(leakage_model, dtype=np.float64)


###############################################################################
# 5) Main example usage
###############################################################################
if __name__ == "__main__":
    ############# (A) Setup ###############
    vcd_folder = "/media/abish/Extreme Pro/Sbox_600/vcds"#vcdssbox"
    clock_name = "SYSCLK_P"
    total_vcds = 600

    # Load plaintexts from file
    path = "/media/abish/Extreme Pro/Sbox_600"#"/media/abish/Extreme Pro/sbox_600/"
    txt_path = os.path.join(path, "plaintexts.txt")
    hex_arrays = []
    with open(txt_path, "r") as file:
        for line in file:
            parts = line.strip().split(":")
            if len(parts) == 2:
                hex_values = parts[1].strip().split()
                hex_arrays.append(hex_values)

    # Convert to 2D array (num_files, bytes_per_line)
    textin_array = []
    for row in hex_arrays:
        textin_array.append([int(x, 16) for x in row])
    textin_array = np.array(textin_array, dtype=np.uint8)
    print("textin_array shape:", textin_array.shape)

    # Example key
    corkey = [0x2b,0x7e,0x15,0x16,0x28,0xae,0xd2,0xa6,0xab,0xf7,0x15,0x88,0x09,0xcf,0x4f,0x3c]
    selected_byte = 0
    chosen_bit = 0  # LSB
    key_guess = corkey[selected_byte]

    plaintext_for_each_trace = textin_array[:, selected_byte]  # shape (600,)
    leakage_model = extract_leakage_model_hd_bit(plaintext_for_each_trace, key_guess, selected_bit=chosen_bit)
    print(f"Built HD model for {total_vcds} traces (bit={chosen_bit}).")

    ############# (B) Parse VCDs in parallel ###############
    print(f"Parsing {total_vcds} VCD files in parallel ...")
    all_parsed = parse_all_vcds_in_parallel(vcd_folder, clock_signal=clock_name, total_files=total_vcds)
    print("Done parsing.\n")

    ############# (C) Build big arrays for each signal ###############
    print("Unifying signals across all VCDs and building arrays...")
    signals_dict = unify_signals_and_build_arrays(all_parsed)
    print(f"Done. Found {len(signals_dict)} unique signals.\n")

    ############# (D) Compute toggle-based correlation for each signal ###############
    toggle_corr_results = []
    leakage_sum_result =[]
    for sig_name, signal_array in tqdm(signals_dict.items(), desc="Toggle correlation"):
        # 1) Build toggles
        toggle_array = build_toggle_array(signal_array)
        # 2) Correlate toggles with leakage model
        leakage_sum = np.multiply(toggle_array, leakage_model)
        corr_toggles = compute_correlation(toggle_array, leakage_model)
        # 3) Store results
        max_val = np.max(np.abs(corr_toggles))
        best_idx = np.argmax(np.abs(corr_toggles))
        leakage_sum_result.append((sig_name, leakage_sum))
        toggle_corr_results.append((sig_name, corr_toggles, max_val, best_idx))

    # Sort by max absolute correlation
    toggle_corr_results.sort(key=lambda x: x[2], reverse=True)

    ############# (E) Plot the correlation vs. clock for each signal ###############
    # WARNING: if you have hundreds of signals, this means hundreds of plots!
    # You may want to limit to top 10 or top 20 signals. Example:
    # for i, (sig_name, corr_toggles, max_val, best_idx) in enumerate(toggle_corr_results[:10]):
    #     ...



    # If you'd like, also print top results in console:
    print("\n=== Top Toggle-Correlated Signals ===")
    for i, (sig_name, corr_arr, max_val, best_idx) in enumerate(toggle_corr_results[:20]):
        print(f"{i+1:2d}) {sig_name}: max abs corr={max_val:.4f} at clock={best_idx}")


In [ ]:
#mem version

import os
import re
import warnings
import numpy as np
from scipy.stats import pearsonr, ConstantInputWarning
from tqdm import tqdm
import matplotlib.pyplot as plt  # for plotting

###############################################################################
# 1) Single-file parser
###############################################################################
def parse_vcd_signals_per_clock(vcd_file, clock_signal_name):
    """
    Parses a VCD and returns a dict: { signal_name: [waveform_values_across_clocks, ...], ... }
    The 'waveform_values_across_clocks' is built whenever we detect a rising edge in the
    clock signal or whenever the clock toggles (depending on your actual logic).

    Note: For large VCDs, returning entire waveforms for all signals can be very big.
          Make sure you have enough memory or simplify further if needed.
    """
    signal_map       = {}
    signal_values    = {}
    signal_waveforms = {}
    clock_id         = None

    in_dumpvars  = False
    current_time = 0

    with open(vcd_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith("$var"):
                parts = line.split()
                if len(parts) >= 5:
                    sig_id = parts[3]
                    name_tokens = parts[4:-1]
                    sig_name = " ".join(name_tokens).strip()

                    signal_map[sig_id] = sig_name
                    signal_values[sig_id] = 'x'
                    if sig_name == clock_signal_name:
                        clock_id = sig_id

                    # Initialize an empty list for this signal’s waveform
                    signal_waveforms[sig_name] = []
                continue

            if line.startswith("$dumpvars"):
                in_dumpvars = True
                continue
            if line.startswith("$end") and in_dumpvars:
                in_dumpvars = False
                continue

            if line.startswith('#'):
                try:
                    current_time = int(line[1:])
                except ValueError:
                    current_time = 0
                continue

            # Parse values
            if in_dumpvars or (current_time >= 0):
                # Single-bit: e.g. "0A", "1A", "xA", etc.
                m_single = re.match(r'([01xXzZ])(\S+)', line)
                if m_single:
                    val, sid = m_single.groups()
                    if sid in signal_values:
                        val = val.lower()
                        val = '0' if (val == 'x' or val == 'z') else val
                        signal_values[sid] = val

                        # Each time the clock changes, we treat that as a "new cycle" sample
                        if sid == clock_id:
                            # For every signal, append the current integer value
                            for s_id, curr_val in signal_values.items():
                                s_name = signal_map[s_id]
                                ivalue = 1 if curr_val == '1' else 0
                                signal_waveforms[s_name].append(ivalue)

                else:
                    # Multi-bit: e.g. "b1010 A"
                    m_bus = re.match(r'b([01xXzZ]+)\s+(\S+)', line)
                    if m_bus:
                        bits_str, sid = m_bus.groups()
                        if sid in signal_values:
                            bits_str = bits_str.lower().replace('x','0').replace('z','0')
                            signal_values[sid] = bits_str

                            if sid == clock_id:
                                # For every signal, append the current integer value
                                for s_id, curr_val in signal_values.items():
                                    s_name = signal_map[s_id]
                                    curr_val = curr_val.lower().replace('x','0').replace('z','0')
                                    ivalue = int(curr_val, 2)
                                    signal_waveforms[s_name].append(ivalue)

    return signal_waveforms


###############################################################################
# 2) Sequential parser over many VCDs
###############################################################################
def parse_all_vcds_sequential(folder, clock_signal, total_files=100):
    """
    Parses VCD files #1..#total_files located in `folder`, named 'vcd1.vcd', 'vcd2.vcd', etc.
    Returns a list (length = total_files) of dicts, each dict is { signal_name : waveform_list }.

    Doing it sequentially avoids big memory blowups from large concurrency.
    """
    filepaths = [os.path.join(folder, f"vcd{i}.vcd") for i in range(1, total_files + 1)]
    all_results = []

    for i, path in enumerate(tqdm(filepaths, desc="Parsing VCDs (sequential)")):
        try:
            parsed = parse_vcd_signals_per_clock(path, clock_signal)
            all_results.append(parsed)
        except Exception as e:
            print(f"Error parsing {path}: {e}")
            all_results.append({})

    return all_results


###############################################################################
# 3) Unify signals + build arrays
###############################################################################
def unify_signals_and_build_arrays(all_parsed):
    """
    all_parsed: list of dicts, each dict: {sig_name -> waveform_list_of_ints}

    Returns a dict {sig_name -> 2D numpy array}, shape = (num_files, num_samples).
    """
    num_vcds = len(all_parsed)
    # Gather all unique signal names
    all_signals = set()
    for d in all_parsed:
        all_signals.update(d.keys())
    all_signals = sorted(list(all_signals))

    signals_dict = {}
    for sig_name in tqdm(all_signals, desc="Unifying signals"):
        waveforms = []
        lengths   = []
        for i in range(num_vcds):
            wave_i = all_parsed[i].get(sig_name, [])
            lengths.append(len(wave_i))
            waveforms.append(wave_i)

        max_len = max(lengths) if lengths else 0
        arr = np.zeros((num_vcds, max_len), dtype=np.int32)

        for i in range(num_vcds):
            wave_i = waveforms[i]
            if len(wave_i) == 0:
                # That file had no data for this signal => remains zeros
                continue
            if len(wave_i) < max_len:
                arr[i, :len(wave_i)] = wave_i
                if len(wave_i) < max_len:
                    print(f"Warning: file #{i+1}, signal '{sig_name}' has {len(wave_i)} samples < {max_len}. Padded with 0.")
            else:
                arr[i, :] = wave_i

        signals_dict[sig_name] = arr

    return signals_dict


###############################################################################
# 4) Correlation function
###############################################################################
def compute_correlationsd(trace_array, leakage_model):
    """
    trace_array: shape (num_traces, num_time)
    leakage_model: shape (num_traces,)
    Returns correlation_over_time: shape (num_time,)
    """
    num_traces, num_time = trace_array.shape
    correlation_over_time = np.zeros(num_time, dtype=float)

    for t in range(num_time):
        column = trace_array[:, t]
        # If everything is constant, pearsonr() can give warnings or NaN
        if np.all(column == column[0]) or np.all(leakage_model == leakage_model[0]):
            correlation_over_time[t] = 0.0
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConstantInputWarning)
                r, _ = pearsonr(column, leakage_model)
                correlation_over_time[t] = r

    return correlation_over_time

def compute_correlation(trace_array, leakage_model):
    """
    Compute the raw dot product (column · leakage_model) for each time sample,
    instead of Pearson's correlation.
    """
    num_traces, num_time = trace_array.shape
    correlation_over_time = np.zeros(num_time, dtype=float)

    for t in range(num_time):
        column = trace_array[:, t]
        # Optional: Skip dot product if all values are constant
        if np.all(column == column[0]) or np.all(leakage_model == leakage_model[0]):
            correlation_over_time[t] = 0.0
        else:
            # Dot product
            correlation_over_time[t] = np.dot(column, leakage_model)

    return correlation_over_time

###############################################################################
# 4.5) Build toggle array
###############################################################################
def build_toggle_array(signal_array):
    """
    signal_array: shape (num_traces, num_clocks)

    Return toggles: shape (num_traces, num_clocks), where:
      toggles[i, t] = +1 if signal_array[i, t] != signal_array[i, t-1], else -1
    The first column toggles[i, 0] is set to -1 by convention.
    """
    num_traces, num_clocks = signal_array.shape
    toggles = np.full((num_traces, num_clocks), -1, dtype=np.int8)

    # Start from clock=1 to compare with clock=0
    for t in range(1, num_clocks):
        toggles[:, t] = np.where(signal_array[:, t] != signal_array[:, t-1], +1, -1)

    return toggles


###############################################################################
# Example S-box & HD model (AES)
###############################################################################
sbox = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]

def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=0):
    """
    Example HD model: For each plaintext, compute a single-bit Hamming Distance
    between input-bit and output-bit of an AES S-box transformation.
    Returns array of +1 / -1 depending on that bit flipping.
    """
    leakage_model = []
    for pt in plaintexts:
        sbox_in  = pt ^ key_byte
        bit_in   = (sbox_in >> selected_bit) & 0x1
        sbox_out = sbox[sbox_in]
        bit_out  = (sbox_out >> selected_bit) & 0x1
        hd_bit   = bit_in ^ bit_out
        leakage_model.append(1 if hd_bit == 1 else -1)
    return np.array(leakage_model, dtype=np.float64)


###############################################################################
# 5) Main example usage
###############################################################################
if __name__ == "__main__":
    ############# (A) Setup ###############
    vcd_folder = "/media/abish/Extreme Pro/Sbox_600/vcds"#vcdssbox"
    clock_name = "SYSCLK_P"
    total_vcds = 600

    # Load plaintexts from file
    path = "/media/abish/Extreme Pro/Sbox_600"#"/media/abish/Extreme Pro/sbox_600/"
    txt_path = os.path.join(path, "plaintexts.txt")

    hex_arrays = []
    with open(txt_path, "r") as file:
        for line in file:
            parts = line.strip().split(":")
            if len(parts) == 2:
                hex_values = parts[1].strip().split()
                hex_arrays.append(hex_values)

    # Convert plaintext lines to 2D array (num_files, bytes_per_line)
    textin_array = []
    for row in hex_arrays:
        textin_array.append([int(x, 16) for x in row])
    textin_array = np.array(textin_array, dtype=np.uint8)
    print("textin_array shape:", textin_array.shape)

    # Example key
    corkey = [0x2b,0x7e,0x15,0x16,0x28,0xae,0xd2,0xa6,0xab,0xf7,0x15,0x88,0x09,0xcf,0x4f,0x3c]
    selected_byte = 0
    chosen_bit    = 0  # e.g., LSB
    key_guess     = corkey[selected_byte]

    # Build HD-based leakage model
    plaintext_for_each_trace = textin_array[:, selected_byte]  # shape (600,)
    leakage_model = extract_leakage_model_hd_bit(plaintext_for_each_trace, key_guess, selected_bit=chosen_bit)
    print(f"Built HD model for {total_vcds} traces (bit={chosen_bit}).")

    ############# (B) Parse VCDs sequentially ###############
    print(f"Parsing {total_vcds} VCD files sequentially...")
    all_parsed = parse_all_vcds_sequential(vcd_folder, clock_signal=clock_name, total_files=total_vcds)
    print("Done parsing.\n")

    ############# (C) Build big arrays for each signal ###############
    print("Unifying signals across all VCDs and building arrays...")
    signals_dict = unify_signals_and_build_arrays(all_parsed)
    print(f"Done. Found {len(signals_dict)} unique signals.\n")

    ############# (D) Compute toggle-based correlation for each signal ###############
    toggle_corr_results = []
    for sig_name, signal_array in tqdm(signals_dict.items(), desc="Toggle correlation"):
        # 1) Build toggles
        toggle_array = build_toggle_array(signal_array)
        # 2) Correlate toggles with leakage model
        corr_toggles = compute_correlation(toggle_array, leakage_model)
        max_val = np.max(np.abs(corr_toggles))
        best_idx = np.argmax(np.abs(corr_toggles))
        # 3) Store results
        toggle_corr_results.append((sig_name, corr_toggles, max_val, best_idx))

    # Sort by max absolute correlation
    toggle_corr_results.sort(key=lambda x: x[2], reverse=True)

    ############# (E) (Optional) Display or plot top signals ###############
    print("\n=== Top Toggle-Correlated Signals ===")
    for i, (sig_name, corr_arr, max_val, best_idx) in enumerate(toggle_corr_results[:20]):
        print(f"{i+1:2d}) {sig_name}: max abs corr = {max_val:.4f} at clock index = {best_idx}")

    # Example: just plot the #1 signal's correlation over time
    if toggle_corr_results:
        top_signal, top_corr, _, _ = toggle_corr_results[0]
        plt.figure()
        plt.title(f"Correlation over time (signal: {top_signal})")
        plt.xlabel("Clock index")
        plt.ylabel("Correlation")
        plt.plot(top_corr)
        plt.show()




In [ ]:
import os
import re
import warnings
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.stats import pearsonr, ConstantInputWarning

# NEW: for progress bars
from tqdm import tqdm

###############################################################################
# 1) Single-file parser
###############################################################################
def parse_vcd_signals_per_clock(vcd_file, clock_signal_name):
    """
    Parses a single VCD file, returning a dictionary:
       signal_name -> list_of_integers_across_clock_edges

    - Each time 'clock_signal_name' toggles, we snapshot *all* signals.
    - Single-bit signals become 0 or 1.
    - Multi-bit signals become an integer by interpreting the binary string
      (with any 'x'/'z' replaced by '0').
    - If a signal never appears, it won't be in the dictionary.
    """

    signal_map       = {}  # signal_id -> signal_name
    signal_values    = {}  # signal_id -> "current" (bit-string or single char)
    signal_waveforms = {}  # signal_name -> list of integer values
    clock_id         = None

    in_dumpvars  = False
    current_time = 0

    with open(vcd_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # 1) Detect variable declarations: $var wire 1 ! clock $end
            if line.startswith("$var"):
                parts = line.split()
                # Example: ["$var", "wire", "1", "A", "my_net", "$end"]
                #   parts[3] = signal_id
                #   parts[4:-1] => name tokens
                if len(parts) >= 5:
                    sig_id = parts[3]
                    name_tokens = parts[4:-1]  # skip "$end"
                    sig_name = " ".join(name_tokens).strip()

                    signal_map[sig_id] = sig_name
                    signal_values[sig_id] = 'x'  # default or unknown
                    if sig_name == clock_signal_name:
                        clock_id = sig_id

                    # Prepare an empty list for the waveform
                    signal_waveforms[sig_name] = []
                continue

            # 2) Look for $dumpvars begin / end
            if line.startswith("$dumpvars"):
                in_dumpvars = True
                continue
            if line.startswith("$end") and in_dumpvars:
                in_dumpvars = False
                continue

            # 3) Detect time steps (#<number>)
            if line.startswith('#'):
                # If you need actual time, parse it here
                time_str = line[1:]
                try:
                    current_time = int(time_str)
                except ValueError:
                    current_time = 0
                continue

            # 4) Parse value changes
            #    We record signal states each time the clock toggles
            if in_dumpvars or (current_time >= 0):
                # A) Single-bit change, e.g. "0A", "1A", "xA"
                match_single = re.match(r'([01xXzZ])(\S+)', line)
                if match_single:
                    val, sid = match_single.groups()
                    if sid in signal_values:
                        # Replace 'x'/'z' with '0'
                        val = val.lower()
                        val = '0' if (val == 'x' or val == 'z') else val
                        signal_values[sid] = val

                        # If clock toggled => new snapshot
                        if sid == clock_id:
                            # For each known signal, store an integer in waveforms
                            for s_id, curr_val in signal_values.items():
                                s_name = signal_map[s_id]
                                # single bit => '0' or '1'
                                # Convert to integer
                                int_val = 1 if curr_val == '1' else 0
                                signal_waveforms[s_name].append(int_val)
                    continue

                # B) Multi-bit change, e.g. "b1010 A", "b1z0B foo"
                match_bus = re.match(r'b([01xXzZ]+)\s+(\S+)', line)
                if match_bus:
                    bits_str, sid = match_bus.groups()
                    if sid in signal_values:
                        # Lower and replace any x/z with 0
                        bits_str = bits_str.lower().replace('x', '0').replace('z', '0')
                        signal_values[sid] = bits_str

                        # If the clock toggled => snapshot
                        if sid == clock_id:
                            for s_id, curr_val in signal_values.items():
                                s_name = signal_map[s_id]
                                # If multi-bit, interpret as binary integer
                                # Single-bit was stored as '0' or '1', so also works
                                curr_val = curr_val.lower().replace('x','0').replace('z','0')
                                int_val = int(curr_val, 2)
                                signal_waveforms[s_name].append(int_val)
                    continue
    # End of file

    return signal_waveforms


###############################################################################
# 2) Parallel wrapper
###############################################################################
def parse_vcd_file_wrapper(vcd_file, clock_signal):
    """Wrapper for parallel usage. Returns dict of {signal_name -> list of int}."""
    return parse_vcd_signals_per_clock(vcd_file, clock_signal)


def parse_all_vcds_in_parallel(folder, clock_signal, total_files=600):
    """
    Parse vcd1.vcd..vcdN.vcd in parallel, returning a list of dicts.
    all_parsed[i] = { signal_name : list_of_int_values_across_clock_edges }
    """
    filepaths = [os.path.join(folder, f"vcd{i}.vcd") for i in range(1, total_files + 1)]
    all_results = [None]*total_files

    with ProcessPoolExecutor() as executor:
        futures_map = {}
        for i, path in enumerate(filepaths):
            fut = executor.submit(parse_vcd_file_wrapper, path, clock_signal)
            futures_map[fut] = i

        # ADDED: progress bar around 'as_completed'
        for fut in tqdm(as_completed(futures_map), total=total_files, desc="Parsing VCDs"):
            i = futures_map[fut]
            try:
                result = fut.result()
                all_results[i] = result
            except Exception as e:
                print(f"Error parsing {filepaths[i]}: {e}")
                all_results[i] = {}

    return all_results


###############################################################################
# 3) Unify signals + build arrays
###############################################################################
def unify_signals_and_build_arrays(all_parsed):
    """
    all_parsed: list of dictionaries (length = num_vcds).
       all_parsed[i] = { signal_name -> list_of_int_values_across_clock_edges }

    We want to find *all* unique signals across these VCDs. Then for each signal,
    gather the waveforms. If lengths differ, pad with 0 (and print a warning).

    Returns:
      signals_dict: { signal_name : 2D numpy array shape (num_vcds, max_length) }
    """
    num_vcds = len(all_parsed)
    # 1) Gather all signal names
    all_signals = set()
    for d in all_parsed:
        all_signals.update(d.keys())
    all_signals = sorted(list(all_signals))

    signals_dict = {}

    # ADDED: show progress for unifying each signal
    for sig_name in tqdm(all_signals, desc="Unifying signals"):
        # gather waveforms from each file
        waveforms = []
        lengths   = []
        for i in range(num_vcds):
            wave_i = all_parsed[i].get(sig_name, [])
            lengths.append(len(wave_i))
            waveforms.append(wave_i)

        max_len = max(lengths)
        # Build 2D array: shape (num_vcds, max_len)
        arr = np.zeros((num_vcds, max_len), dtype=np.int32)
        for i in range(num_vcds):
            wave_i = waveforms[i]
            if len(wave_i) < max_len:
                # Pad with 0
                if len(wave_i) > 0:
                    arr[i, :len(wave_i)] = wave_i
                if len(wave_i) > 0 and len(wave_i) < max_len:
                    print(f"Warning: file #{i+1}, signal '{sig_name}' has "
                          f"{len(wave_i)} edges < {max_len}. Padded with 0.")
            else:
                arr[i,:] = wave_i
        signals_dict[sig_name] = arr

    return signals_dict

###############################################################################
# Additional code to build a leakage model, etc.
###############################################################################
sbox = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]

def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=0):
    """
    Single-bit Hamming Distance between S-box input & output, returning +1 if bit flips, -1 otherwise.
    plaintexts: 1D array of shape (N,)
    """
    leakage_model = []
    for pt in plaintexts:
        sbox_in = pt ^ key_byte
        bit_in = (sbox_in >> selected_bit) & 0x1
        sbox_out = sbox[sbox_in]
        bit_out = (sbox_out >> selected_bit) & 0x1
        hd_bit = bit_in ^ bit_out
        leakage_model.append(1 if hd_bit == 1 else -1)
    return np.array(leakage_model, dtype=np.float64)

###############################################################################
# 4) Correlation function
###############################################################################
def compute_correlation(trace_array, leakage_model):
    """
    trace_array: shape (num_traces, num_time_samples)
    leakage_model: shape (num_traces,)
      => For each time sample t, compute correlation across all traces.
    Returns correlation_over_time of shape (num_time_samples,).
    """
    num_traces, num_time = trace_array.shape
    correlation_over_time = np.zeros(num_time, dtype=float)

    for t in range(num_time):
        column = trace_array[:, t]  # shape (num_traces,)
        # Handle constant warnings:
        if np.all(column == column[0]) or np.all(leakage_model == leakage_model[0]):
            correlation_over_time[t] = 0.0
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConstantInputWarning)
                r, _ = pearsonr(column, leakage_model)
                correlation_over_time[t] = r

    return correlation_over_time

###############################################################################
# 4.5) Build toggle array
###############################################################################
def build_toggle_array(signal_array):
    """
    signal_array: shape (num_traces, num_clocks)
      - Each row is one trace, each column is the signal value at that clock.
    Returns:
      toggles: shape (num_traces, num_clocks),
          +1 if signal_array[i, t] != signal_array[i, t-1], else -1
      For t=0, we define toggles[i, 0] = -1 by convention.
    """
    num_traces, num_clocks = signal_array.shape
    toggles = np.full((num_traces, num_clocks), -1, dtype=np.int8)

    for t in range(1, num_clocks):
        toggles[:, t] = np.where(signal_array[:, t] != signal_array[:, t-1], +1, -1)

    return toggles

###############################################################################
# 5) Main example usage
###############################################################################
if __name__ == "__main__":
    ############# (A) Setup ###############
    vcd_folder = "/media/abish/Extreme Pro/sbox_600/vcds"
    clock_name = "SYSCLK_P"
    total_vcds = 600

    # Example: load plaintexts from file
    path = "/media/abish/Extreme Pro/sbox_600/"
    txt_path = os.path.join(path, "plaintexts.txt")
    hex_arrays = []
    with open(txt_path, "r") as file:
        for line in file:
            parts = line.strip().split(":")
            if len(parts) == 2:
                hex_values = parts[1].strip().split()
                hex_arrays.append(hex_values)

    # Convert to 2D array (num_files, bytes_per_line)
    textin_array = []
    for row in hex_arrays:
        textin_array.append([int(x, 16) for x in row])
    textin_array = np.array(textin_array, dtype=np.uint8)
    print("textin_array shape:", textin_array.shape)

    # Example key
    corkey = [0x2b,0x7e,0x15,0x16,0x28,0xae,0xd2,0xa6,0xab,0xf7,0x15,0x88,0x09,0xcf,0x4f,0x3c]
    selected_byte = 0
    chosen_bit = 0  # LSB
    key_guess = corkey[selected_byte]

    # Build the leakage model (HD bit) using the selected byte
    plaintext_for_each_trace = textin_array[:, selected_byte]   # shape (600,)
    leakage_model = extract_leakage_model_hd_bit(plaintext_for_each_trace, key_guess, selected_bit=chosen_bit)
    print(f"Built HD model for {total_vcds} traces (bit={chosen_bit}).")

    ############# (B) Parse VCDs in parallel ###############
    print(f"Parsing {total_vcds} VCD files in parallel ...")
    all_parsed = parse_all_vcds_in_parallel(vcd_folder, clock_signal=clock_name, total_files=total_vcds)
    print("Done parsing.\n")

    ############# (C) Build big arrays for each signal ###############
    print("Unifying signals across all VCDs and building arrays...")
    signals_dict = unify_signals_and_build_arrays(all_parsed)
    print(f"Done. Found {len(signals_dict)} unique signals.\n")

    ############# (D1) For each signal, compute correlation (normal) ###############
    results = []  # will store tuples of (signal_name, best_corr_value, best_clock_idx)

    for sig_name, trace_array in tqdm(signals_dict.items(), desc="Computing correlation"):
        # trace_array: shape (600, #clock_edges_for_this_signal)
        corr_over_time = compute_correlation(trace_array, leakage_model)
        # find maximum correlation in absolute value
        max_corr_value = np.max(np.abs(corr_over_time))
        best_clock = np.argmax(np.abs(corr_over_time))
        # Store the signed correlation at that clock
        best_signed_corr = corr_over_time[best_clock]
        results.append((sig_name, best_signed_corr, best_clock))

    # Sort signals by absolute maximum correlation descending
    results.sort(key=lambda x: abs(x[1]), reverse=True)

In [ ]:
############# (D) Compute toggle-based correlation for each signal ###############

def compute_correlation(trace_array, leakage_model):
    """
    Compute the raw dot product (column · leakage_model) for each time sample,
    instead of Pearson's correlation.
    """
    num_traces, num_time = trace_array.shape
    correlation_over_time = np.zeros(num_time, dtype=float)

    for t in range(num_time):
        column = trace_array[:, t]
        # Optional: Skip dot product if all values are constant
        if np.all(column == column[0]) or np.all(leakage_model == leakage_model[0]):
            correlation_over_time[t] = 0.0
        else:
            # Dot product
            correlation_over_time[t] = np.dot(column, leakage_model)

    return correlation_over_time


toggle_corr_results = []
for sig_name, signal_array in tqdm(signals_dict.items(), desc="Toggle correlation"):
    # 1) Build toggles
    toggle_array = build_toggle_array(signal_array)
    # 2) Correlate toggles with leakage model
    corr_toggles = compute_correlation(toggle_array, leakage_model)
    # 3) Store results
    max_val = np.max(np.abs(corr_toggles))
    best_idx = np.argmax(np.abs(corr_toggles))
    toggle_corr_results.append((sig_name, corr_toggles, max_val, best_idx))

# Sort by max absolute correlation
toggle_corr_results.sort(key=lambda x: x[2], reverse=True)

############# (E) Plot the correlation vs. clock for each signal ###############
# WARNING: if you have hundreds of signals, this means hundreds of plots!
# You may want to limit to top 10 or top 20 signals. Example:
# for i, (sig_name, corr_toggles, max_val, best_idx) in enumerate(toggle_corr_results[:10]):
#     ...

for (sig_name, corr_toggles, max_val, best_idx) in toggle_corr_results:
    # Make a new figure
    plt.figure()
    plt.plot(corr_toggles)
    plt.title(f"Toggle correlation vs. clock: {sig_name}\n"
              f"Max corr={max_val:.4f} at clock={best_idx}")
    plt.xlabel("Clock index")
    plt.ylabel("Correlation")
    plt.show()

# If you'd like, also print top results in console:
print("\n=== Top Toggle-Correlated Signals ===")
for i, (sig_name, corr_arr, max_val, best_idx) in enumerate(toggle_corr_results[:20]):
    print(f"{i+1:2d}) {sig_name}: max abs corr={max_val:.4f} at clock={best_idx}")


In [ ]:
leakage_contributions = {}

for sig_name, signal_array in signals_dict.items():
    toggle_array = build_toggle_array(signal_array)  # shape (600, 228)
    weighted = toggle_array * leakage_model[:, np.newaxis]  # shape (600, 228)
    leakage_sum_per_clock = np.sum(weighted, axis=0)        # shape (228,)
    leakage_contributions[sig_name] = weighted

In [ ]:
def rank_signals_by_leakage_contribution(toggle_dict, leakage_model):
    """
    Given a dict of toggle arrays for multiple signals (toggle_dict),
    convert each signal's toggles to +1 or -1, then multiply by leakage_model
    and compute the sum. Finally, return signals ranked by that sum (descending).

    Parameters:
    -----------
    toggle_dict   : dict of {signal_name: 1D or 2D array}
                    If each entry is a 1D array of toggles (length T),
                    it must match len(leakage_model) = T.
    leakage_model : 1D array of length T

    Returns:
    --------
    ranked_signals : list of (signal_name, leakage_sum) sorted by descending leakage_sum
    """
    leakage_contributions = {}

    for signal, toggles in toggle_dict.items():
        # toggles could be shape (num_traces, num_time) or just (num_time,).
        # Here we assume toggles is 1D for simplicity. If not, adapt accordingly.
        if toggles.shape[-1] != len(leakage_model):
            continue  # Skip signals with mismatched length

        # Convert toggle rates: +1 if toggle>0 else -1
        # If toggles is 1D -> shape (num_time,); else adapt with np.where if 2D
        toggles_1d = toggles if toggles.ndim == 1 else toggles.mean(axis=0)
        # Above we took the average across traces if 2D.
        # You could also sum or pick another approach.

        modified_toggle_rates = np.where(toggles_1d > 0, +1, -1)

        # Multiply by the leakage model pointwise, then sum across time
        leakage_sum = np.sum(modified_toggle_rates * leakage_model)

        # Store result
        leakage_contributions[signal] = leakage_sum

    # Sort signals by their total leakage contribution (descending)
    ranked_signals = sorted(leakage_contributions.items(), key=lambda x: x[1], reverse=True)
    return ranked_signals
toggle_dict = {}
for sig_name, signal_array in signals_dict.items():
    toggle_array = build_toggle_array(signal_array)
    # Possibly reduce it to 1D by summing or averaging across VCDs
    # For a single trace per signal: shape is (num_clocks,)
    # For multiple traces: shape is (num_traces, num_clocks)
    toggle_dict[sig_name] = toggle_array
ranked = rank_signals_by_leakage_contribution(toggle_dict, leakage_model)


In [ ]:
print(ranked)

In [ ]:
###############################################################################
# (F) Compute overall (mean) toggles across all signals
###############################################################################
print("Computing mean toggles across all signals for each clock ...")
start_ps      = 268350000
clock_step_ps = 50000

num_signals = len(signals_dict)
accum_toggles = None

# We'll reuse the same 'build_toggle_array' function used earlier
for sig_name, signal_array in signals_dict.items():
    # signal_array has shape (num_traces, num_clocks)
    tarr = build_toggle_array(signal_array)  # shape (num_traces, num_clocks)

    # Accumulate toggles across signals
    if accum_toggles is None:
        # Convert to float so we don't lose precision when summing
        accum_toggles = tarr.astype(np.float64)
    else:
        accum_toggles += tarr

# Compute the average toggles by dividing by the number of signals
mean_toggles = accum_toggles / num_signals  # shape (num_traces, num_clocks)
print("mean_toggles shape:", mean_toggles.shape)

###############################################################################
# (G) Correlate the design-wide mean toggles with the leakage model
###############################################################################
print("Computing correlation for overall design toggles vs. leakage model ...")
overall_corr = compute_correlation(mean_toggles, leakage_model)
time_ps = start_ps + clock_step_ps * np.arange(num_clocks)
plt.figure()
# Plot corr vs time
plt.plot(time_ps, abs(overall_corr))

# overall_corr is a 1D array of length num_clocks
# You can now plot it vs. clock cycle index or time

#plt.plot(abs(overall_corr))
plt.title("Overall Mean Toggle Correlation vs. Clock Cycle")
plt.xlabel("Clock Cycle Index")
plt.ylabel("Correlation")
plt.savefig("overall_mean_toggle_correlation.png", bbox_inches="tight", transparent=True)
plt.show()

print("Done. The plot 'overall_mean_toggle_correlation.png' shows the design-wide correlation.")


In [ ]:
###############################################################################
# Create and plot a single figure with all signals, X-axis in picoseconds
###############################################################################
start_ps      = 268350000
clock_step_ps = 50000

plt.figure()
for (sig_name, corr_toggles, max_val, best_idx) in toggle_corr_results:
    num_clocks = corr_toggles.shape[0]

    # Build time axis: 268350000, 268400000, 268450000, ...
    time_ps = start_ps + clock_step_ps * np.arange(num_clocks)

    # Plot corr vs time
    plt.plot(time_ps, abs(corr_toggles), label=sig_name)

plt.title("All Signals Toggle Correlation in One Chart (vs. Time)")
plt.xlabel("Time (ps)")
plt.ylabel("Correlation")
# plt.legend()  # May be huge if you have hundreds of signals
plt.savefig("Toggle_correlation.pdf", bbox_inches="tight", transparent=True)
plt.savefig("Toggle_correlation.svg", bbox_inches="tight", transparent=True)
plt.savefig("Toggle_correlation.png", bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
threshold = 0.2

start_ps      = 268350000
clock_step_ps = 50000

plt.figure()

for (sig_name, corr_toggles, max_val, best_idx) in toggle_corr_results:
    num_clocks = corr_toggles.shape[0]

    # Build time axis in ps
    time_ps = start_ps + clock_step_ps * np.arange(num_clocks)

    # Check if this signal's max correlation exceeds the threshold
    if max_val > threshold:
        label_str = sig_name
        if sig_name =="u_top_gen_regfile_ff.register_file_i_rf_reg_q[13] [0]":
            label_str ="register_file_i_rf_reg_q[13] [0]"
    else:
        label_str = None

    plt.plot(time_ps, corr_toggles, label=label_str)

plt.title("All Signals Toggle Correlation with AES stat bit 0 (vs. Time)")
plt.xlabel("Time (ps)")
plt.ylabel("Correlation")
plt.legend()
plt.savefig("Toggle2_correlation.pdf", bbox_inches="tight", transparent=True)
plt.savefig("Toggle2_correlation.svg", bbox_inches="tight", transparent=True)
plt.savefig("Toggle2_correlation.png", bbox_inches="tight", transparent=True)
# Only signals labeled above will appear in the legend

plt.show()


In [ ]:
threshold = 0.2

start_ps      = 268350000
clock_step_ps = 50000

all_corr_arrays = []  # This will collect absolute correlation vectors for all signals

plt.figure()

for (sig_name, corr_toggles, max_val, best_idx) in toggle_corr_results:
    num_clocks = corr_toggles.shape[0]
    time_ps = start_ps + clock_step_ps * np.arange(num_clocks)

    # Accumulate the absolute correlation for computing the mean later
    all_corr_arrays.append(np.abs(corr_toggles))

    # Only label signals if their max correlation exceeds threshold
    if max_val > threshold:
        label_str = sig_name
        # Example of short labeling a certain signal
        if sig_name == "u_top_gen_regfile_ff.register_file_i_rf_reg_q[13] [0]":
            label_str = "register_file_i_rf_reg_q[13] [0]"
    else:
        label_str = None

    # Plot each signal individually (absolute value)
    plt.plot(time_ps, np.abs(corr_toggles), label=label_str)

plt.title("All Signals Toggle Correlation with AES stat bit 0 (vs. Time)")
plt.xlabel("Time (ps)")
plt.ylabel("Correlation")
plt.legend()
plt.savefig("Toggle_correlation.pdf", bbox_inches="tight", transparent=True)
plt.savefig("Toggle_correlation.svg", bbox_inches="tight", transparent=True)
plt.savefig("Toggle_correlation.png", bbox_inches="tight", transparent=True)
plt.show()

#############################################################################
# Now compute and plot the mean correlation across ALL signals vs. time
#############################################################################

# Convert the list of correlation arrays to a single NumPy array:
# Shape will be [num_signals, num_clocks].
all_corr_arrays = np.array(all_corr_arrays)
mean_corr = np.mean(all_corr_arrays, axis=0)

# time_ps is the same indexing as before, so just reuse the length of mean_corr
num_clocks = mean_corr.shape[0]
time_ps = start_ps + clock_step_ps * np.arange(num_clocks)

plt.figure()
plt.plot(time_ps, mean_corr, label='Mean Correlation of All Signals')
plt.title("Mean Toggle Correlation Across All Signals (vs. Time)")
plt.xlabel("Time (ps)")
plt.ylabel("Mean Correlation")
plt.legend()
plt.savefig("Mean_Toggle_correlation.pdf", bbox_inches="tight", transparent=True)
plt.savefig("Mean_Toggle_correlation.svg", bbox_inches="tight", transparent=True)
plt.savefig("Mean_Toggle_correlation.png", bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
# After you build 'toggle_corr_results' which contains:
#   [(sig_name, corr_toggles, max_val, best_idx), ...]
#   where 'corr_toggles' is a 1D array of correlation vs. clock.

import matplotlib.pyplot as plt

# Make a single figure
plt.figure()

# Plot each signal's correlation on the same axes
for (sig_name, corr_toggles, max_val, best_idx) in toggle_corr_results:
    plt.plot(corr_toggles, label=sig_name)

plt.title("All Signals Toggle Correlation in One Chart")
plt.xlabel("Clock index")
plt.ylabel("Correlation")
#plt.legend()  # May get very large if you have many signals
plt.show()

## VCD Process

In [ ]:
import re
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Parse signal maps from the first VCD file
def parse_vcd_signalmaps(vcd_file, func):
    signal_map = {}  # Map signal IDs to names
    with open(vcd_file, 'r') as file:
        vart = False
        for line in file:
            line = line.strip()
            if not line:
                continue
            if not vart:
                if line.startswith("$scope module " + func + " $end"):
                    vart = True
                continue
            # Parse signal declarations
            if line.startswith("$var"):
                parts = line.split()
                signal_id = parts[3]
                signal_name = parts[4] + parts[5] if len(parts) > 6 else parts[4]
                signal_map[signal_id] = signal_name.strip()
            elif line.startswith("$upscope"):
                break
    return signal_map

# Parse VCD within a time interval
def parse_vcd_within_time_interval(vcd_file, signal_map, time_start, time_end):
    toggle_counts = {signal: 0 for signal in signal_map.values()}  # Initialize toggle counts
    signal_values = {signal: '0' for signal in signal_map.keys()}  # Initialize signal values
    current_time = 0

    with open(vcd_file, 'r') as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            # Detect time steps
            if line.startswith('#'):
                current_time = int(line[1:])
                continue

            # Process signal value changes within the time interval
            if current_time >= time_start and current_time <= time_end:
                match = re.match(r'([01zx])(\S+)', line)
                if match:
                    value, signal_id = match.groups()
                    if signal_id in signal_map:
                        prev_value = signal_values[signal_id]
                        signal_values[signal_id] = value
                        if prev_value != value:
                            signal_name = signal_map[signal_id]
                            toggle_counts[signal_name] += 1

    return toggle_counts

# Process all VCD files
def process_vcd_files(vcd_folder, func, time_start, time_end):
    vcd_file = os.path.join(vcd_folder, f'vcd1.vcd')
    signal_map = parse_vcd_signalmaps(vcd_file, func)  # Parse signal map once
    toggle_rates = {signal: [] for signal in signal_map.values()}  # Initialize toggle rates

    # Loop through all VCD files with progress bar
    for i in tqdm(range(1, 423), desc="Processing VCD Files"):
        vcd_file = os.path.join(vcd_folder, f'vcd{i}.vcd')
        toggle_counts = parse_vcd_within_time_interval(vcd_file, signal_map, time_start, time_end)

        # Append toggle counts to rates
        for signal, count in toggle_counts.items():
            toggle_rates[signal].append(count)

    return toggle_rates

# Rank signals by their correlation with the leakage model
# Rank signals by their correlation with the leakage model
#def rank_signals_by_leakage(toggle_rates, leakage_model):
 #   correlations = {}
 #   for signal, counts in toggle_rates.items():
 #       if len(counts) != len(leakage_model):
 #           continue  # Skip if lengths don't match

        # Check for constant values or zero variance
 #       if np.var(counts) == 0 or np.var(leakage_model) == 0:
 #           correlations[signal] = 0  # Assign zero correlation for invalid cases
 #           continue

        # Compute Pearson correlation
 #       corr, _ = pearsonr(counts, leakage_model)
 #       if np.isnan(corr):  # Handle NaN correlations
 #           correlations[signal] = 0
 #       else:
  #          correlations[signal] = abs(corr)  # Use absolute correlation
#
    # Sort signals by correlation in descending order
 #   ranked_signals = sorted(correlations.items(), key=lambda x: x[1], reverse=True)
 #   return ranked_signals
import numpy as np

def rank_signals_by_leakage_contribution(toggle_rates, leakage_model):
    leakage_contributions = {}

    for signal, counts in toggle_rates.items():
        if len(counts) != len(leakage_model):
            continue  # Skip signals with mismatched lengths

        # Convert toggle rates: +1 for tg > 0, -1 for tg <= 0
        modified_toggle_rates = [1 if tg > 0 else -1 for tg in counts]

        # Multiply modified toggle rates with the leakage model and calculate the sum
        leakage_sum = np.sum(np.multiply(modified_toggle_rates, leakage_model))

        # Store the leakage sum for the signal
        leakage_contributions[signal] = leakage_sum

    # Sort signals by leakage contribution (descending)
    ranked_signals = sorted(leakage_contributions.items(), key=lambda x: x[1], reverse=True)
    return ranked_signals

# Visualization of ranked signals
def visualize_ranked_signals(ranked_signals, top_n=10):
    top_signals = ranked_signals[:top_n]
    signals, correlations = zip(*top_signals)

    plt.barh(signals, correlations)
    plt.xlabel("Correlation with Leakage Model")
    plt.title(f"Top {top_n} Signals by Leakage Contribution")
    plt.gca().invert_yaxis()
    plt.show()

# Main
if __name__ == "__main__":
    vcd_folder = path+"/vcds"  # Folder containing VCD files
    function_name = "uut"  # Top module
    #time_start = 268587500  # Start time in ns
    #time_end = 277462500   # End time in ns


    # Process VCD files
    toggle_rates = process_vcd_files(vcd_folder, function_name, time_start, time_end)


##########
    # Rank signals
    ranked_signals = rank_signals_by_leakage_contribution(toggle_rates, leakage_model)

    # Print ranked signals
    print("\nRanked Signals by Leakage Contribution:")
    for rank, (signal, corr) in enumerate(ranked_signals, start=1):
        print(f"Rank {rank}: Signal = {signal}, Correlation = {corr:.5f}")
        if rank >10:
            break

# Visualize the top-ranked signals
    visualize_ranked_signals(ranked_signals, top_n=50)


## Netlist process (get gates accourding to signal) + compute Impact factor

In [ ]:
import re
analysis_correlation_limitation = 10
total_Power = 9.28787e-05
# Preprocess signal names
def preprocess_signal_name(signal):
    return signal.replace("][", "] [")
# Calculate the ranking based on Correlation x Average Power
def rank_signals_by_impact(signal_gate_mapping):
    ranked_signals = []

    for signal, data in signal_gate_mapping.items():
        # Skip signals with no calculated impact
        #if data["Impact"] > 0:
            # Handle NaN correlation by assuming it as 0
        correlation = data["correlation"]
        if correlation is None:
            correlation = 0  # Treat NaN as 0

            # Calculate the impact value
        impact_value = correlation * data["Impact"]

            # Find the gate instance name with the "output" role
        output_instance = None
        for gate in data["gates"]:
            if gate["role"] == "output":
                output_instance = gate["instance_name"]
                break

            # Add the signal, impact value, and output instance to the ranking list
        ranked_signals.append({
                "signal": signal,
                "impact_value": impact_value,
                "output_instance": output_instance,
                "correlation": correlation,  # Include correlation for debugging purposes
            })

    # Sort signals by impact value in descending order
    ranked_signals.sort(key=lambda x: x["impact_value"], reverse=True)

    return ranked_signals

# Debugging Output: Verify Intermediate Data
def print_debug_data(signal_gate_mapping):
    print("\nSignal Data (Before Ranking):")
    for signal, data in signal_gate_mapping.items():
        print(f"Signal: {signal}, Correlation: {data['correlation']}, Impact: {data['Impact']:.6e}")
        for gate in data["gates"]:
            print(f"  Gate: {gate['gate_name']}, Instance: {gate['instance_name']}, Role: {gate['role']}")

# Debugging Output: Verify Ranked Signals
def print_ranked_signals(ranked_signals):
    print("\nRanked Signals by Impact Value:")
    for rank, entry in enumerate(ranked_signals, start=1):
        print(f"Rank {rank}: Signal = {entry['signal']}, "
              f"Impact Value = {entry['impact_value']:.6e}, "
              f"correlation = {entry['correlation']}, "
              f"Output Instance = {entry['output_instance']}")

def find_signal_in_netlist(signal, netlist_file):
    connections = []
    with open(netlist_file, 'r') as file:
        lines = file.readlines()

    # Create a regex pattern to match the exact signal name
    escaped_signal = re.escape(signal).replace(r'\[', r'\s*\[').replace(r'\]', r'\s*\]')
    signal_pattern = re.compile(rf'(?<!\w){escaped_signal}(?!\w)')  # Match the signal exactly

    for i, line in enumerate(lines):
        # Skip wire declarations
        if line.strip().startswith("wire"):
            continue

        # Check if the signal is in the current line
        if signal_pattern.search(line):  # Use regex search for exact match
            # Backtrack to find the start of the gate (starting with SC8T_)
            start_idx = i
            while start_idx > 0 and not lines[start_idx].strip().startswith("SC8T_"):
                start_idx -= 1

            # Forward track to find the end of the gate (ending with ;)
            end_idx = i
            while end_idx < len(lines) and ";" not in lines[end_idx]:
                end_idx += 1

            # Capture the full gate definition
            gate_definition = "".join(lines[start_idx:end_idx + 1]).strip()
            gate_definition_cleaned = re.sub(r'\s+', ' ', gate_definition)

            connections.append(gate_definition_cleaned)

    return connections

# Parse gate definitions from the technology file
def parse_gate_definitions(tech_file):
    gates_dict = {}
    module_def_regex = re.compile(r'module\s+(\S+)\s*\(([^)]*)\);')
    input_regex = re.compile(r'input\s+([^;]+);')
    output_regex = re.compile(r'output\s+([^;]+);')

    with open(tech_file, 'r') as vf:
        verilog_content = vf.read()

    for match in module_def_regex.finditer(verilog_content):
        gate_name, ports = match.groups()
        ports = [p.strip() for p in ports.split(',')]

        # Extract inputs and outputs
        inputs = []
        outputs = []
        start_idx = verilog_content.find(match.group(0)) + len(match.group(0))
        module_body = verilog_content[start_idx:verilog_content.find("endmodule", start_idx)]

        for input_match in input_regex.finditer(module_body):
            inputs.extend(input_match.group(1).split(','))

        for output_match in output_regex.finditer(module_body):
            outputs.extend(output_match.group(1).split(','))

        gates_dict[gate_name] = {
            "name": gate_name,
            "inputs": [i.strip() for i in inputs],
            "outputs": [o.strip() for o in outputs],
            "ports": ports
        }
    return gates_dict

# Parse port-to-signal mapping
def parse_connection_ports(connection):
    port_signal_regex = re.findall(r'\.(\w+)\s*\(\s*(.*?)\s*\)', connection)
    return {port: signal.replace("\\", "").strip() for port, signal in port_signal_regex}

# Extract power for each instance from the power report
def extract_power_per_instance(power_report_file):
    power_per_instance = {}
    with open(power_report_file, 'r') as file:
        for line in file:
            if re.match(r'^\s*[0-9eE.+-]+\s+[0-9eE.+-]+\s+[0-9eE.+-]+\s+[0-9eE.+-]+\s+/.+', line):
                parts = line.split()
                total_power = float(parts[-2])  # Extract total power
                instance_name = parts[-1]  # Extract instance name
                power_per_instance[instance_name] = total_power
    return power_per_instance

# Analyze high-correlation signals
def analyze_high_correlation_signals(ranked_signals, netlist_file, tech_file, power_report_file):
    gates_dict = parse_gate_definitions(tech_file)
    power_data = extract_power_per_instance(power_report_file)
    signal_gate_mapping = {}

    for signal, correlation in ranked_signals:
        if abs(correlation) <= analysis_correlation_limitation:
            continue
        processed_signal = preprocess_signal_name(signal)
        connections = find_signal_in_netlist(processed_signal, netlist_file)

        signal_gate_mapping[signal] = {
            "correlation": correlation,
            "gates": [],
            "Impact": 0
        }

        instance_powers = []
        for connection in connections:
            # Extract gate instantiation name
            gate_match = re.search(r'(SC8T_\S+)', connection)
            gate_matchxc = re.search(r'(SC8T_\S+)\s+(\S+)\(', connection)
            if gate_matchxc == None:
                gate_matchxc = re.search(r'(SC8T_\S+)\s+(\S+)\s*\(', connection)
            if gate_match:
                gate_name = gate_match.group(1)
                if gate_name in gates_dict:
                    gate_info = gates_dict[gate_name]
                    instance_namef = gate_matchxc.group(2)
                    instance_name = instance_namef.replace("\\", "")


                    # Parse port-to-signal mapping
                    port_signal_map = parse_connection_ports(connection)

                    # Determine role
                    role = "unknown"
                    for port, signal_name in port_signal_map.items():
                        if signal_name == processed_signal:
                            if port in gate_info["inputs"]:
                                role = "input"
                            elif port in gate_info["outputs"]:
                                role = "output"
                            break

                    # Append gate information
                    signal_gate_mapping[signal]["gates"].append({
                        "gate_name": gate_name,
                        "gate_info": gate_info,
                        "instance_name": instance_name,
                        "connection": connection,
                        "role": role
                    })
                    if role=="output":
                        for power_entry in power_data.keys():  # Iterate through power data entries
                            power_instance_name = power_entry.split('/')[-1]  # Get the part after the last '/'
                            if instance_name == power_instance_name:
                            # Instance name matches, retrieve the power value
                                instance_powers.append(power_data[power_entry])
                                break

                    # After collecting powers, calculate average


        # Calculate average power for the signal
        if instance_powers:
            average_power = sum(instance_powers) / len(instance_powers)  # Sum up all the instance powers
            signal_gate_mapping[signal]["Impact"] = average_power / total_Power



    return signal_gate_mapping

# Example Usage
if __name__ == "__main__":
    ranked_signals_new=[]
    netlist_file = path +"/"+"PROACT_topa_netlist.v"
    tech_file = path +"/"+"GF22FDX_SC8T_116CPP_BASE_DDC28UH.v"
    power_report_file = path +"/"+"Power_per_instance.txt"

    signal_gate_mapping = analyze_high_correlation_signals(ranked_signals, netlist_file, tech_file, power_report_file)

    # Print results
    ranked_signals_by_impact = rank_signals_by_impact(signal_gate_mapping)

    print_ranked_signals(ranked_signals_by_impact)


# new

In [ ]:


# Print the resulting array
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.stats import pearsonr
import numpy as np
from scipy.stats import pearsonr, ConstantInputWarning
import warnings
###############################################################################
# Original (working) parse function - UNCHANGED
###############################################################################
def parse_vcd_at_clock_changes(vcd_file, clock_signal_name):
    """
    Parses a VCD file, detecting all signals. Whenever the clock (identified by 'clock_signal_name')
    changes, record transitions of every other signal. Returns:
      signal_waves: dict with {signal_name: wave_string}
      transition_counts: {
         'N_transition_to_0': [...],
         'N_transition_to_1': [...],
         'Total_N_transition': [...]
      }
    The length of 'Total_N_transition' is the number of complete clock cycles (beyond the first edge).
    """
    signal_map = {}
    signal_values = {}
    previous_values = {}
    signal_waveforms = {}
    transition_counts = {
        'N_transition_to_0': [],
        'N_transition_to_1': [],
        'Total_N_transition': []
    }
    clock_id = None
    in_dumpvars = False
    current_time = 0

    with open(vcd_file, 'r') as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            # 1) Detect variable declarations ($var)
            if line.startswith("$var"):
                parts = line.split()
                if len(parts) >= 5:
                    signal_id = parts[3]
                    signal_name_tokens = parts[4:-1]  # skip the "$end"
                    signal_name = " ".join(signal_name_tokens).strip()

                    # Store in maps
                    signal_map[signal_id] = signal_name
                    signal_values[signal_id] = 'x'
                    previous_values[signal_id] = 'x'
                    # If it's the clock signal
                    if signal_name == clock_signal_name:
                        clock_id = signal_id
                    # Prepare waveform storage
                    signal_waveforms[signal_name] = []
                continue

            # 2) Look for $dumpvars begin / end
            if line.startswith("$dumpvars"):
                in_dumpvars = True
                continue
            if line.startswith("$end") and in_dumpvars:
                in_dumpvars = False
                continue

            # 3) Time steps (#<number>)
            if line.startswith('#'):
                time_val = line[1:]
                try:
                    current_time = int(time_val)
                except ValueError:
                    current_time = 0
                continue

            # 4) Parse value changes
            if in_dumpvars or (current_time >= 0):
                # A) Single-bit: e.g. "0\"", "1!"
                match_single = re.match(r'([01zx])(\S+)', line)
                if match_single:
                    value, sid = match_single.groups()
                    if sid in signal_values:
                        signal_values[sid] = value

                        # If it's the clock toggling => new cycle
                        if sid == clock_id:
                            # 1) Snapshot all signals
                            for s_id, s_val in signal_values.items():
                                s_name = signal_map[s_id]
                                signal_waveforms[s_name].append(s_val)

                            # 2) If we have >= 2 clock edges, measure toggles
                            if len(signal_waveforms[clock_signal_name]) > 1:
                                n_to_0 = 0
                                n_to_1 = 0
                                total_transitions = 0
                                for other_id, curr_val in signal_values.items():
                                    if other_id == clock_id:
                                        continue
                                    prev_val = previous_values[other_id]
                                    if prev_val != curr_val:
                                        total_transitions += 1
                                        if curr_val == '0':
                                            n_to_0 += 1
                                        elif curr_val == '1':
                                            n_to_1 += 1
                                    previous_values[other_id] = curr_val

                                transition_counts['N_transition_to_0'].append(n_to_0)
                                transition_counts['N_transition_to_1'].append(n_to_1)
                                transition_counts['Total_N_transition'].append(total_transitions)
                            else:
                                # First clock edge => init previous_values
                                for other_id, curr_val in signal_values.items():
                                    previous_values[other_id] = curr_val
                    continue

                # B) Multi-bit
                match_bus = re.match(r'b([01zx]+)\s+(\S+)', line)
                if match_bus:
                    bits, sid = match_bus.groups()
                    if sid in signal_values:
                        signal_values[sid] = bits

                        # If clock_id, handle multi-bit clock, etc. (not used here)
                    continue
            # end if in_dumpvars or current_time >= 0
        # end for line
    # end with open

    # Build wave strings
    signal_waves = {}
    for sname, values in signal_waveforms.items():
        wave_str = ''.join(values)
        signal_waves[sname] = wave_str

    return signal_waves, transition_counts


###############################################################################
# 2) Parallel wrapper to parse multiple VCD files
###############################################################################
def parse_vcd_file_wrapper(vcd_file, clock_signal):
    """
    Simple wrapper that calls parse_vcd_at_clock_changes and returns
    only the 'Total_N_transition' array for convenience.
    """
    _, transition_counts = parse_vcd_at_clock_changes(vcd_file, clock_signal)
    toggles = np.array(transition_counts['Total_N_transition'], dtype=np.float64)
    return toggles  # shape (#clock_edges - 1,)


def parse_all_vcds_in_parallel(folder, clock_signal, total_files=600):
    """
    Parse vcd1.vcd..vcdN.vcd in parallel, returning a list of toggles arrays.
    all_traces[i] = toggles from vcd{i+1}.vcd
    """
    filepaths = [os.path.join(folder, f"vcd{i}.vcd") for i in range(1, total_files + 1)]
    all_traces = [None] * total_files

    with ProcessPoolExecutor(max_workers=1) as executor:
        futures_map = {}
        for idx, path in enumerate(filepaths):
            fut = executor.submit(parse_vcd_file_wrapper, path, clock_signal)
            futures_map[fut] = idx

        for fut in tqdm(as_completed(futures_map), total=total_files, desc="Parsing VCDs"):
            idx = futures_map[fut]
            try:
                toggles_array = fut.result()
                all_traces[idx] = toggles_array
            except Exception as e:
                print(f"Error parsing file {filepaths[idx]}: {e}")

    return all_traces


###############################################################################
# 3) Single-bit HD model for AES (bit 0)
###############################################################################
sbox = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]

def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=0):
    """
    Single-bit Hamming Distance between S-box input & output, returning +1 if bit flips, -1 otherwise.
    plaintexts: 1D array of shape (N,)
    """
    leakage_model = []
    for pt in plaintexts:
        sbox_in = pt ^ key_byte
        bit_in = (sbox_in >> selected_bit) & 0x1
        sbox_out = sbox[sbox_in]
        bit_out = (sbox_out >> selected_bit) & 0x1
        hd_bit = bit_in ^ bit_out
        leakage_model.append(1 if hd_bit == 1 else -1)
    return np.array(leakage_model, dtype=np.float64)


###############################################################################
# 4) Compute correlation over "time" = clock edge index
###############################################################################
def compute_correlation(trace_array, leakage_model):
    """
    trace_array: shape (num_traces, num_time_samples)
    leakage_model: shape (num_traces,)
      => For each time sample (clock edge) t, compute correlation across all traces.
    Returns correlation_over_time of shape (num_time_samples,)
    """
    num_traces, num_time = trace_array.shape
    correlation_over_time = np.zeros(num_time, dtype=np.float64)
    for t in range(num_time):
        trace_t = trace_array[:, t]  # shape (num_traces,)
        r, _ = pearsonr(trace_t, leakage_model)
        correlation_over_time[t] = r
    return correlation_over_time


def compute_correlation(trace_array, leakage_model):
    num_traces, num_time = trace_array.shape
    correlation_over_time = np.zeros(num_time, dtype=float)

    for t in range(num_time):
        trace_t = trace_array[:, t]  # (num_traces,)

        # If trace_t or leakage_model is constant, set correlation to nan
        if (np.all(trace_t == trace_t[0])) or (np.all(leakage_model == leakage_model[0])):
            correlation_over_time[t] = 0
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConstantInputWarning)
                r, _ = pearsonr(trace_t, leakage_model)
                correlation_over_time[t] = r

    return correlation_over_time


###############################################################################
# 5) Main
###############################################################################
if __name__ == "__main__":
    ############################
    # A) Load plaintexts
    ############################
    # Initialize an array to store the extracted values

    path="/media/abish/Extreme Pro/sbox_600/"#"courses/sca101/res/"

    txt_path = os.path.join(path, "plaintexts.txt")
    hex_arrays = []

    with open(txt_path, "r") as file:
        for line in file:
            parts = line.strip().split(":")
            if len(parts) == 2:
                hex_values = parts[1].strip().split()
                hex_arrays.append(hex_values)

    # Convert to 2D array (num_files, bytes_per_line)
    textin_array = []
    for row in hex_arrays:
        textin_array.append([int(x, 16) for x in row])
    textin_array = np.array(textin_array, dtype=np.uint8)
    print("textin_array shape:", textin_array.shape)

    ############################
    # B) Parse 500 VCDs in parallel
    ############################
    vcd_folder = os.path.join(path, "vcdssbox")
    total_files = textin_array.shape[0]  # e.g. 500
    clock_signal = "SYSCLK_P"

    print(f"Parsing {total_files} VCD files in parallel ...")
    all_traces_list = parse_all_vcds_in_parallel(vcd_folder, clock_signal, total_files=total_files)
    print("Done. Collected toggles arrays.\n")

    # Check all have the same length
    lengths = [len(arr) for arr in all_traces_list if arr is not None]
    if len(set(lengths)) != 1:
        print("Error: Not all VCD files have the same number of clock edges!")
        for i, arr in enumerate(all_traces_list):
            print(f"File {i+1}: length={len(arr)}")
        raise SystemExit("Aborting due to mismatch in clock cycle lengths.")
    num_clocks = lengths[0]
    print(f"Confirmed each file has {num_clocks} clock edges.\n")

    # Stack them => shape (num_files, num_clocks)
    trace_array = np.vstack(all_traces_list)
    print("trace_array shape:", trace_array.shape)



In [ ]:



correlation_over_time = abs(compute_correlation(trace_array, hd_model))

########################################
# E) Plot the correlation trace
########################################
plt.figure(figsize=(8,4))
plt.plot(correlation_over_time)
plt.xlabel("Clock Edge Index")
plt.ylabel("Correlation")
plt.title(f"Correlation vs. Clock Edge (Byte={selected_byte}, Bit={chosen_bit})")
plt.grid(True)
plt.show()
print("Done.")

In [ ]:
import os
import re
import gc
import pickle
import warnings
import numpy as np
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.stats import pearsonr, ConstantInputWarning
import matplotlib.pyplot as plt  # for plotting


###############################################################################
# 1) Single-file parser (Clock-Based) with Time Window
###############################################################################
def parse_vcd_clock_based(
    vcd_file,
    clock_signal_name="SYSCLK_P",
    start_time_ps=0,
    stop_time_ps=float('inf'),
    clock_duration_ps=100_000,  # e.g. 100 ns
    toggles_per_cycle=2
):
    """
    1) Estimate number of clock toggles in [start_time_ps..stop_time_ps).
       N_clock_est = (stop_time_ps - start_time_ps)/clock_duration_ps * toggles_per_cycle
    2) Pre-allocate arrays of shape (N_clock_est,) for each net => signal_arrays[sig_name].
    3) Keep last_known_value for each net. As we parse lines:
       - if we see #<time_ps> outside the time window => skip
       - if net changes, we update last_known_value[sig_name]
       - if the clock toggles => current_clock_idx++ => fill all signal_arrays[...] with last_known_value
    4) Trim arrays if we used fewer toggles than estimated.

    Returns => dict { sig_name : np.array( shape=(actual_clocks,) ) }
    """
    first= False
    total_ps = max(0, stop_time_ps - start_time_ps)
    N_clock_est = int((total_ps / clock_duration_ps) * toggles_per_cycle)
    if N_clock_est < 1:
        N_clock_est = 1  # at least 1

    signal_arrays    = {}
    last_known_value = {}
    signal_defs      = {}

    current_clock_idx = -1
    clock_id          = None

    current_time_ps   = 0
    last_clock_val    = None
    in_dumpvars       = False

    def ensure_signal_array(sig_name):
        if sig_name not in signal_arrays:
            signal_arrays[sig_name]    = np.zeros(N_clock_est, dtype=object)
            last_known_value[sig_name] = 0

    with open(vcd_file, 'r') as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue

            # parse definitions
            if line.startswith("$var"):
                parts = line.split()
                if len(parts) >= 5:
                    var_id   = parts[3]
                    sig_name = " ".join(parts[4:-1]).strip()
                    # register net
                    signal_defs[var_id] = sig_name
                    ensure_signal_array(sig_name)
                    if sig_name == clock_signal_name:
                        clock_id = var_id
                continue

            if line.startswith("$dumpvars"):
                in_dumpvars = True
                continue
            if line.startswith("$end") and in_dumpvars:
                in_dumpvars = False
                continue

            if line.startswith('#'):
                try:
                    current_time_ps = int(line[1:])
                    if first==False:
                        #print(f"time started with={current_time_ps}") #268350000time
                        start=current_time_ps
                        first=True
                except ValueError:
                    current_time_ps = 0
                continue

            # skip if outside time window
            if not (start_time_ps <= current_time_ps < stop_time_ps):
                continue

            # single-bit => "1A", "0A", ...
            m_single = re.match(r'^([01xXzZ])(\S+)$', line)
            if m_single:
                val, var_id = m_single.groups()
                val = val.lower()
                if val in ['x','z']:
                    val = '0'
                if var_id in signal_defs:
                    sig_name = signal_defs[var_id]
                    last_known_value[sig_name] = 1 if val=='1' else 0

                if var_id == clock_id:
                    new_val = 1 if val=='1' else 0
                    if (last_clock_val is not None) and (new_val != last_clock_val):
                        current_clock_idx += 1
                        # fill arrays if we haven't exceeded capacity
                        if current_clock_idx < N_clock_est:
                            for s_name in signal_arrays.keys():
                                signal_arrays[s_name][current_clock_idx] = last_known_value[s_name]
                    last_clock_val = new_val
                continue

            # multi-bit => "b1010 A"
            m_bus = re.match(r'^b([01xXzZ]+)\s+(\S+)$', line)
            if m_bus:
                bits_str, var_id = m_bus.groups()
                bits_str = bits_str.lower().replace('x','0').replace('z','0')
                val_int = int(bits_str, 2)

                if var_id in signal_defs:
                    sig_name = signal_defs[var_id]
                    last_known_value[sig_name] = val_int

                if var_id == clock_id:
                    if (last_clock_val is not None) and (val_int != last_clock_val):
                        current_clock_idx += 1
                        if current_clock_idx < N_clock_est:
                            for s_name in signal_arrays.keys():
                                signal_arrays[s_name][current_clock_idx] = last_known_value[s_name]
                    last_clock_val = val_int

    # trim
    actual_clocks = max(0, current_clock_idx+1)
    for s_name, arr in signal_arrays.items():
        if actual_clocks < N_clock_est:
            signal_arrays[s_name] = arr[:actual_clocks]

    return signal_arrays


###############################################################################
# 2) Parallel parse wrapper
###############################################################################
def parse_vcd_clock_based_wrapper(args):
    """
    For use with ProcessPoolExecutor. 'args' => (
       vcd_file,
       clock_signal,
       start_time_ps,
       stop_time_ps,
       clock_duration_ps,
       toggles_per_cycle
    )
    """
    (vcd_file, clock_signal_name, start_ps, stop_ps, clock_dur, toggles_per_cycle) = args
    return parse_vcd_clock_based(
        vcd_file=vcd_file,
        clock_signal_name=clock_signal_name,
        start_time_ps=start_ps,
        stop_time_ps=stop_ps,
        clock_duration_ps=clock_dur,
        toggles_per_cycle=toggles_per_cycle
    )


def parse_all_vcds_in_parallel(
    folder,
    clock_signal="SYSCLK_P",
    total_files=70,
    max_workers=8,
    start_time_ps=0,
    stop_time_ps=float('inf'),
    clock_duration_ps=100_000,  # 100 ns
    toggles_per_cycle=2
):
    """
    Parallel approach:
      1) For each file => parse_vcd_clock_based(...) with user-specified times, clock dur, toggles/cycle
      2) Return a list of length = total_files, each item => { sig_name : array (actual_clocks,) }
    """
    filepaths = [os.path.join(folder, f"vcd{i}.vcd") for i in range(1, total_files+1)]
    results = [None]*total_files

    tasks = []
    for i, path in enumerate(filepaths):
        tasks.append((path, clock_signal, start_time_ps, stop_time_ps, clock_duration_ps, toggles_per_cycle))

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        fut_map = {}
        for i,arg_tuple in enumerate(tasks):
            fut = executor.submit(parse_vcd_clock_based_wrapper, arg_tuple)
            fut_map[fut] = i

        for fut in tqdm(as_completed(fut_map), total=total_files, desc="Parsing VCDs"):
            idx = fut_map[fut]
            try:
                net_arrays = fut.result()  # => dict { sig_name : np.array(...) }
                results[idx] = net_arrays
            except Exception as e:
                print(f"Error parsing {filepaths[idx]} => {e}")
                results[idx] = {}

    return results


###############################################################################
# 3) Example main
###############################################################################
if __name__ == "__main__":
    folder = "/media/abish/Extreme Pro/sbox_600/vcdssbox"#vcds"
    clock_signal_name = "SYSCLK_P"
    total_files = 600
    start_ps = 268350000  # e.g. skip first 1,000,000 ps 268350000time
    stop_ps  = 279750000 # e.g. process until 10,000,000 ps


    # We'll parse in parallel using the clock-based approach
    print(f"Parallel parse of {total_files} VCDs, clock={clock_signal_name}")
    print(f"Time window => [{start_ps}..{stop_ps})")

    results_all = parse_all_vcds_in_parallel(
        folder=folder,
        clock_signal=clock_signal_name,
        total_files=total_files,
        max_workers=8,
        start_time_ps=start_ps,
        stop_time_ps=stop_ps,
        clock_duration_ps=100_000,  # 100 ns
        toggles_per_cycle=2
    )

    print("Done parsing parallel.\n")

    # 'results_all[i]' => dict { sig_name : np.array of shape (actual_clocks_i,) }

    # Optionally save
   # with open("vcd_clockbased_results600.pkl", "wb") as f:
     #   pickle.dump(results_all, f, protocol=pickle.HIGHEST_PROTOCOL)

    #print("\nAll done. 'vcd_clockbased_results.pkl' has clock-based arrays for each net in each file.")

In [ ]:
import os
import re
import gc
import pickle
import warnings
import numpy as np
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.stats import pearsonr, ConstantInputWarning
import matplotlib.pyplot as plt

###############################################################################
# Let's assume you already have:
#   1) hd_model: np.array of shape (600,) with your hypothetical leakage model
#   2) results_all: a list of length 600
#      - results_all[i] is a dict of ~8344 nets
#      - each net => np.array of shape (227,) containing 0 or 1
#   3) You want to transform the 0/1 data into:
#        [ net[0], then +1 if bit changed from previous clock else -1, ... ]
#   4) Then compute correlation between each net (across all 600 files) and hd_model
#   5) Sum correlations across nets at each clock, take absolute value, and plot.
###############################################################################

# Example placeholders for demonstration:
# hd_model = np.random.randint(0, 2, size=600)  # shape (600,)
# results_all = [...]
#   ^ In your real code, you already have these from parse_all_vcds_in_parallel

# Just define your timeline:
start_ps      = 268350000
clock_step_ps = 50000

###############################################################################
# 1) Transform the 0/1 arrays into transitions.
###############################################################################
warnings.filterwarnings("ignore", category=ConstantInputWarning)

print("Step 1: Transform raw 0/1 data into (+1, -1) transitions...")
for i in tqdm(range(len(results_all)), desc="Transforming each file"):
    net_dict = results_all[i]  # dict: { net_name: array(227,) with 0/1 }
    for net_name, net_values in net_dict.items():
        # Create an 'object'-dtype array to be safe if signals are large:
        transformed = np.empty_like(net_values, dtype=object)

        # The first value remains the same (0 or 1)
        transformed[0] = net_values[0]

        # For each subsequent clock, store +1 if it changed, else -1
        for clk in range(1, len(net_values)):
            if net_values[clk] != net_values[clk - 1]:
                transformed[clk] = 1
            else:
                transformed[clk] = -1

        # Store back
        net_dict[net_name] = transformed


###############################################################################
# 2) Compute correlation (net by net, clock by clock) across 600 files
###############################################################################
all_net_names = list(results_all[0].keys())      # e.g. 8344 signals
num_nets      = len(all_net_names)
num_clocks    = len(results_all[0][all_net_names[0]])  # 227

print("Step 2: Computing correlations...")
correlations  = np.zeros((num_nets, num_clocks), dtype=float)

for net_idx, net_name in tqdm(enumerate(all_net_names), total=num_nets, desc="Correlating nets"):
    # Gather data for this net from all 600 files => shape (600, 227), dtype=object
    net_data = np.array([results_all[f][net_name] for f in range(len(results_all))], dtype=object)

    # For each clock sample, do a Pearson correlation with hd_model
    for clk in range(num_clocks):
        samples_clock = net_data[:, clk]  # shape (600,) of +1/-1 or 0/1 for the first element
        # Check if it's constant:
        if len(np.unique(samples_clock)) <= 1:
            # All values identical => correlation is undefined, set to 0
            corr_val = 0.0
        else:
            # Convert to float for scipy
            corr_val, _ = pearsonr(samples_clock.astype(float), hd_model.astype(float))

        correlations[net_idx, clk] = corr_val


###############################################################################
# 3) Sum correlations across nets, take absolute value, and plot vs. time
###############################################################################
print("Step 3: Summing correlations and plotting...")
corr_sums     = np.sum(correlations, axis=0)    # shape (227,)
corr_sums_abs = np.abs(corr_sums)

# Build the time axis: clock k => start_ps + k * clock_step_ps
time_ps = start_ps + clock_step_ps * np.arange(num_clocks)

# Plot
plt.figure(figsize=(8, 6))
plt.plot(time_ps, corr_sums_abs, label="|Sum of Correlations|")
plt.xlabel("Time (ps)")
plt.ylabel("Absolute Sum of Correlations")
plt.title("Correlation vs. Time")
plt.legend()
plt.show()

print("Done.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from tqdm import tqdm

# Suppose you already have:
#   hd_model: shape (600,)
#   results_all: list of length 600
#       results_all[i]: dict with ~8344 nets
#           each net => np.array( shape=(227,) ) with values 0 or 1
start_ps      = 268350000
clock_step_ps = 50000

##############################################################################
# 1) Transform the 0/1 arrays into “first-value + transitions” (+1 if changed, -1 otherwise)
##############################################################################
print("Transforming raw 0/1 data into transitions (+1 or -1)...")
for i in tqdm(range(len(results_all)), desc="Step 1/3: Transform arrays"):
    net_dict = results_all[i]  # dict { net_name : array(227,) of 0/1 }
    for net_name, net_values in net_dict.items():
        # net_values is shape (227,) with 0/1
        # Make an 'object'-dtype array so Python int can handle large values:
        transformed = np.empty_like(net_values, dtype=object)

        # Keep the first value as is (0 or 1)
        transformed[0] = net_values[0]

        # For each subsequent clock, store +1 if it changed from the previous clock, else -1
        for clk in range(1, len(net_values)):
            if net_values[clk] != net_values[clk - 1]:
                transformed[clk] = 1
            else:
                transformed[clk] = -1

        # Store back into the dict
        net_dict[net_name] = transformed

##############################################################################
# 2) Compute correlation across the 600 files for each net and each clock
##############################################################################
print("Computing correlations...")
all_net_names = list(results_all[0].keys())    # e.g. 8344 signals
num_nets      = len(all_net_names)
num_clocks    = len(results_all[0][all_net_names[0]])  # 227

# We'll store correlation results as float:
correlations  = np.zeros((num_nets, num_clocks), dtype=float)

for net_idx, net_name in tqdm(enumerate(all_net_names), total=num_nets, desc="Step 2/3: Correlate each net"):
    # Gather this net's data from all 600 files: shape (600, 227), dtype=object
    net_data = np.array([results_all[f][net_name] for f in range(len(results_all))], dtype=object)

    # For each clock sample, do a Pearson correlation with hd_model
    for clk in range(num_clocks):
        # net_data[:, clk] => shape (600,)
        corr_val, _ = pearsonr(net_data[:, clk].astype(float), hd_model.astype(float))
        correlations[net_idx, clk] = corr_val

##############################################################################
# 3) Sum correlations across nets, take absolute value, and plot vs time
##############################################################################
print("Summing correlations and plotting...")
corr_sums     = np.sum(correlations, axis=0)   # shape (227,) => sum over net dimension
corr_sums_abs = np.abs(corr_sums)

# Build x-axis in ps:
time_ps = start_ps + clock_step_ps * np.arange(num_clocks)

# Plot
plt.figure(figsize=(8, 6))
plt.plot(time_ps, corr_sums_abs, label="|Sum of Correlations|")
plt.xlabel("Time (ps)")
plt.ylabel("Absolute Sum of Correlations")
plt.title("Correlation vs. Time")
plt.legend()
plt.show()


In [ ]:
##############################################################################
# 3) Sum correlations across nets, take absolute value, and plot vs time
##############################################################################
print("Summing correlations and plotting...")
corr_sums     = np.sum(correlations, axis=0)   # shape (227,) => sum over net dimension
corr_sums_abs = np.abs(corr_sums)

# Build x-axis in ps:
time_ps = start_ps + clock_step_ps * np.arange(num_clocks)

# Plot
plt.figure(figsize=(8, 6))
plt.plot(time_ps, corr_sums_abs, label="|Sum of Correlations|")
plt.xlabel("Time (ps)")
plt.ylabel("Absolute Sum of Correlations")
plt.title("Correlation vs. Time")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# --- Inputs ---
# hd_model: shape (600,)
# results_all: list of length 600
#   results_all[i]: dict with ~8344 nets
#       each net => np.array( shape=(227,) ) with values 0 or 1
start_ps      = 268350000
clock_step_ps = 50000

# 1) Transform the 0/1 arrays to “first‐value + transitions” (+1 if changed, −1 if not).
for i in range(len(results_all)):            # i in [0..599]
    net_dict = results_all[i]               # dict { net_name : array(227,) }
    for net_name, net_values in net_dict.items():
        # net_values is 0/1 of length 227
        transformed = np.empty_like(net_values, dtype=object)
        transformed[0] = net_values[0]      # keep the first value as 0 or 1
        for clk in range(1, len(net_values)):
            if net_values[clk] != net_values[clk - 1]:
                transformed[clk] = 1
            else:
                transformed[clk] = -1
        net_dict[net_name] = transformed    # store back

# 2) Compute correlation (net by net, clock by clock).
all_net_names = list(results_all[0].keys())    # all signals (e.g. 8344)
num_nets      = len(all_net_names)
num_clocks    = len(results_all[0][all_net_names[0]])  # 227

# correlations[net_idx, clock_idx] => correlation with hd_model
correlations = np.zeros((num_nets, num_clocks))

for net_idx, net_name in enumerate(all_net_names):
    # net_data has shape (600, 227): each row is one file, each column is a clock sample
    net_data = np.array([results_all[f][net_name] for f in range(len(results_all))], dtype=int)
    # For each clock index, correlate net_data[:, clock] with hd_model
    for clk in range(num_clocks):
        corr_val, _ = pearsonr(net_data[:, clk], hd_model)
        correlations[net_idx, clk] = corr_val

# 3) Sum correlations across all nets at each clock, take abs, and plot vs time.
corr_sums     = np.sum(correlations, axis=0)   # shape (227,) => sum over net dimension
corr_sums_abs = np.abs(corr_sums)

# Build your x-axis in ps: clock 0 => start_ps, clock 1 => start_ps + clock_step_ps, ...
time_ps = start_ps + clock_step_ps * np.arange(num_clocks)

# Plot
plt.figure(figsize=(8,6))
plt.plot(time_ps, corr_sums_abs, label="|Sum of Correlations|")
plt.xlabel("Time (ps)")
plt.ylabel("Absolute Sum of Correlations")
plt.title("Correlation vs. Time")
plt.legend()
plt.show()


In [ ]:


def compute_time_average_correlations(
    results_all,hd_model

):
    net_names_set = set()
    for d in results_all:
        net_names_set.update(d.keys())
    net_names = sorted(net_names_set)
    num_files  = len(results_all)



    target_array = hd_model

    corrs = []

    # Add a progress bar around net_names loop
    for net_name in tqdm(net_names, leave=False):
        # net_profile => shape=(rng,)
        net_profile = np.zeros(num_files, dtype=np.float64)

        # For each time in [idx1..idx2)
        for local_idx, time_idx in enumerate(range(idx1, idx2)):
            # gather from all files => shape=(num_files,), then average
            row_vals = np.zeros(num_files, dtype=np.float64)
            for f_idx in range(num_files):
                arr = results_all[f_idx].get(net_name, np.array([], dtype=int))
                # if time_idx < len(arr), use arr[time_idx], else 0
                if time_idx < len(arr):
                    row_vals[f_idx] = arr[time_idx]
                else:
                    row_vals[f_idx] = 0.0
            net_profile[local_idx] = np.mean(row_vals)

        # correlation => net_profile vs target_array
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConstantInputWarning)
            if (np.all(net_profile == net_profile[0])
                or np.all(target_array == target_array[0])):
                r_val = 0.0
            else:
                r_val, _ = pearsonr(net_profile, target_array)

        corrs.append((net_name, r_val))

    # sort by abs correlation
    corrs.sort(key=lambda x: abs(x[1]), reverse=True)
    results_per_byte[byte_idx] = corrs

    return results_per_byte

correlation_over_time = abs(compute_time_average_correlations(results_all, hd_model))

########################################
# E) Plot the correlation trace
########################################
plt.figure(figsize=(8,4))
plt.plot(correlation_over_time)
plt.xlabel("Clock Edge Index")
plt.ylabel("Correlation")
plt.title(f"Correlation vs. Clock Edge (Byte={selected_byte}, Bit={chosen_bit})")
plt.grid(True)
plt.show()
print("Done.")

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm.notebook import trange
from Crypto.Cipher import AES  # PyCryptodome library for AES verification
import time
from scipy.stats import pearsonr, norm, ConstantInputWarning
import re
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm


###############################################################################
# 1) Single-file parser
###############################################################################
def parse_vcd_signals_per_clock(vcd_file, clock_signal_name):
    signal_map       = {}
    signal_values    = {}
    signal_waveforms = {}
    clock_id         = None

    in_dumpvars  = False
    current_time = 0

    with open(vcd_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith("$var"):
                parts = line.split()
                if len(parts) >= 5:
                    sig_id = parts[3]
                    name_tokens = parts[4:-1]
                    sig_name = " ".join(name_tokens).strip()

                    signal_map[sig_id] = sig_name
                    signal_values[sig_id] = 'x'
                    if sig_name == clock_signal_name:
                        clock_id = sig_id

                    signal_waveforms[sig_name] = []
                continue

            if line.startswith("$dumpvars"):
                in_dumpvars = True
                continue
            if line.startswith("$end") and in_dumpvars:
                in_dumpvars = False
                continue

            if line.startswith('#'):
                try:
                    current_time = int(line[1:])
                except ValueError:
                    current_time = 0
                continue

            if in_dumpvars or (current_time >= 0):
                # Single-bit: "0A", "1A", "xA" ...
                m_single = re.match(r'([01xXzZ])(\S+)', line)
                if m_single:
                    val, sid = m_single.groups()
                    if sid in signal_values:
                        val = val.lower()
                        val = '0' if (val == 'x' or val == 'z') else val
                        signal_values[sid] = val

                        if sid == clock_id:
                            for s_id, curr_val in signal_values.items():
                                s_name = signal_map[s_id]
                                int_val = 1 if curr_val == '1' else 0
                                signal_waveforms[s_name].append(int_val)
                else:
                    # Multi-bit: "b1010 A"
                    m_bus = re.match(r'b([01xXzZ]+)\s+(\S+)', line)
                    if m_bus:
                        bits_str, sid = m_bus.groups()
                        if sid in signal_values:
                            bits_str = bits_str.lower().replace('x','0').replace('z','0')
                            signal_values[sid] = bits_str

                            if sid == clock_id:
                                for s_id, curr_val in signal_values.items():
                                    s_name = signal_map[s_id]
                                    curr_val = curr_val.lower().replace('x','0').replace('z','0')
                                    int_val = int(curr_val, 2)
                                    signal_waveforms[s_name].append(int_val)

    return signal_waveforms


###############################################################################
# 2) Parallel wrapper
###############################################################################
def parse_vcd_file_wrapper(vcd_file, clock_signal):
    return parse_vcd_signals_per_clock(vcd_file, clock_signal)


def parse_all_vcds_in_parallel(folder, clock_signal, total_files=600,mxworker):
    from concurrent.futures import ProcessPoolExecutor, as_completed
    filepaths = [os.path.join(folder, f"vcd{i}.vcd") for i in range(1, total_files + 1)]
    all_results = [None]*total_files

    with ProcessPoolExecutor(max_workers=mxworker) as executor:
        futures_map = {}
        for i, path in enumerate(filepaths):
            fut = executor.submit(parse_vcd_file_wrapper, path, clock_signal)
            futures_map[fut] = i

        for fut in tqdm(as_completed(futures_map), total=total_files, desc="Parsing VCDs"):
            i = futures_map[fut]
            try:
                result = fut.result()
                all_results[i] = result
            except Exception as e:
                print(f"Error parsing {filepaths[i]}: {e}")
                all_results[i] = {}

    return all_results


###############################################################################
# 3) Unify signals + build arrays
###############################################################################
def unify_signals_and_build_arrays(all_parsed):
    num_vcds = len(all_parsed)
    all_signals = set()
    for d in all_parsed:
        all_signals.update(d.keys())
    all_signals = sorted(list(all_signals))

    signals_dict = {}
    for sig_name in tqdm(all_signals, desc="Unifying signals"):
        waveforms = []
        lengths   = []
        for i in range(num_vcds):
            wave_i = all_parsed[i].get(sig_name, [])
            lengths.append(len(wave_i))
            waveforms.append(wave_i)

        max_len = max(lengths)
        arr = np.zeros((num_vcds, max_len), dtype=np.int32)
        for i in range(num_vcds):
            wave_i = waveforms[i]
            if len(wave_i) < max_len:
                if len(wave_i) > 0:
                    arr[i, :len(wave_i)] = wave_i
                if len(wave_i) > 0 and len(wave_i) < max_len:
                    print(f"Warning: file #{i+1}, signal '{sig_name}' has "
                          f"{len(wave_i)} edges < {max_len}. Padded with 0.")
            else:
                arr[i, :] = wave_i
        signals_dict[sig_name] = arr

    return signals_dict


###############################################################################
# 4) Correlation function
###############################################################################
def compute_correlation(trace_array, leakage_model):
    num_traces, num_time = trace_array.shape
    correlation_over_time = np.zeros(num_time, dtype=float)

    for t in range(num_time):
        column = trace_array[:, t]
        if np.all(column == column[0]) or np.all(leakage_model == leakage_model[0]):
            correlation_over_time[t] = 0.0
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConstantInputWarning)
                r, _ = pearsonr(column, leakage_model)
                correlation_over_time[t] = r

    return correlation_over_time


###############################################################################
# 4.5) Build toggle array
###############################################################################
def build_toggle_array(signal_array):
    """
    signal_array: shape (num_traces, num_clocks)

    Return toggles: shape (num_traces, num_clocks), where
      toggles[i, t] = +1 if signal_array[i, t] != signal_array[i, t-1] else -1
    The first column toggles[i, 0] is set to -1 by convention.
    """
    num_traces, num_clocks = signal_array.shape
    toggles = np.full((num_traces, num_clocks), -1, dtype=np.int8)

    for t in range(1, num_clocks):
        toggles[:, t] = np.where(signal_array[:, t] != signal_array[:, t-1], +1, -1)

    return toggles


###############################################################################
# Additional: Example s-box + HD model
###############################################################################
sbox = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]
###############################################################################
# Single-bit HD model extraction
###############################################################################

def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=0):
    leakage_model = []
    for pt in plaintexts:
        sbox_in = pt ^ key_byte
        bit_in = (sbox_in >> selected_bit) & 0x1
        sbox_out = sbox[sbox_in]
        bit_out = (sbox_out >> selected_bit) & 0x1
        hd_bit = bit_in ^ bit_out
        leakage_model.append(1 if hd_bit == 1 else -1)
    return np.array(leakage_model, dtype=np.float64)





def aes_internal(inputdata, key):
    return sbox[inputdata ^ key]

HW = [bin(n).count("1") for n in range(0, 256)]


def mean(X):
    return np.sum(X, axis=0)/len(X)

def std_dev(X, X_bar):
    return np.sqrt(np.sum((X-X_bar)**2, axis=0))

def cov(X, X_bar, Y, Y_bar):
    return np.sum((X-X_bar)*(Y-Y_bar), axis=0)
# Function to compute KL Divergence for normal distributions
def kl_divergence(mu_x, sigma_x, mu_y, sigma_y):
    return ((mu_x - mu_y)**2) / (2 * sigma_y**2) + (sigma_x**2) / (2 * sigma_y**2) - 0.5 + np.log(sigma_y / sigma_x)


def pearson_confidence_interval(x, y, alpha=0.05):
    # Calculate Pearson's correlation coefficient using scipy.stats.pearsonr
    r, p_value = pearsonr(x, y)
    n = len(x)

    # Fisher's z-transform to approximate normality
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)

    # Calculate the critical z-value
    z_critical = norm.ppf(1 - alpha / 2)

    # Compute confidence interval in z-space
    z_interval = z + np.array([-1, 1]) * z_critical * se

    # Transform back to correlation scale
    r_interval = np.tanh(z_interval)

    return r, r_interval, p_value




# Find significant leakage time intervals
def find_leakage_time_interval(correlation_over_time, threshold=0.1):
    return np.where(abs(correlation_over_time) > threshold)[0]

###############################################################################
# Example usage
###############################################################################
if __name__ == "__main__":
    number_of_traces =100
    hex_arrays = []
    path="/media/abish/Extreme Pro/vcdfull/"#sbox_600/"#"courses/sca101/res/"
# Open the file and process each line
    with open(path+"plaintexts.txt", "r") as file:
      for line in file:
        # Split the line by spaces and extract the hex values (ignoring the label before the colon)
            parts = line.strip().split(":")
           if len(parts) == 2:
                hex_values = parts[1].strip().split()
                hex_arrays.append(hex_values)

##trace_arrayx= np.array(data_arrays)[:, :-1] #remove last value of traces

    textin_arraycc=hex_arrays[:number_of_traces]
    textin_array = [[int(value, 16) for value in hex_array] for hex_array in textin_arraycc] # convert to hex
    print(f"textin_array array shape: {np.shape(textin_array)}")
    tistart= time.time()
    vcd_folder = "/media/abish/Extreme Pro/vcdfull/vcds"#vcdssbox"
    path = "/media/abish/Extreme Pro/vcdfull"#"/media/abish/Extreme Pro/sbox_600/"
    clock_name = "SYSCLK_P"
    total_vcds = 100
    mxworker=5
    # Load plaintexts from file

    txt_path = os.path.join(path, "plaintexts.txt")
    hex_arrays = []
    with open(txt_path, "r") as file:
        for line in file:
            parts = line.strip().split(":")
            if len(parts) == 2:
                hex_values = parts[1].strip().split()
                hex_arrays.append(hex_values)

    # Convert to 2D array (num_files, bytes_per_line)
    textin_array = []
    for row in hex_arrays:
        textin_array.append([int(x, 16) for x in row])
    textin_array = np.array(textin_array, dtype=np.uint8)
    print("textin_array shape:", textin_array.shape)

    # Example key
    corkey = [0x2b,0x7e,0x15,0x16,0x28,0xae,0xd2,0xa6,0xab,0xf7,0x15,0x88,0x09,0xcf,0x4f,0x3c]
    selected_byte = 0
    chosen_bit = 0  # LSB
    key_guess = corkey[selected_byte]

    plaintext_for_each_trace = textin_array[:, selected_byte]  # shape (600,)
    leakage_model = extract_leakage_model_hd_bit(plaintext_for_each_trace, key_guess, selected_bit=chosen_bit)
    print(f"Built HD model for {total_vcds} traces (bit={chosen_bit}).")

    ############# (B) Parse VCDs in parallel ###############
    print(f"Parsing {total_vcds} VCD files in parallel ...")
    all_parsed = parse_all_vcds_in_parallel(vcd_folder, clock_signal=clock_name, total_files=total_vcds,mxworker)
    print("Done parsing.\n")

    ############# (C) Build big arrays for each signal ###############
    print("Unifying signals across all VCDs and building arrays...")
    signals_dict = unify_signals_and_build_arrays(all_parsed)
    print(f"Done. Found {len(signals_dict)} unique signals.\n")

    ############# (D) Compute toggle-based correlation for each signal ###############
    toggle_corr_results = []
    leakage_sum_result =[]
    for sig_name, signal_array in tqdm(signals_dict.items(), desc="Toggle correlation"):
        # 1) Build toggles
        toggle_array = build_toggle_array(signal_array)
        # 2) Correlate toggles with leakage model
        leakage_sum = np.multiply(toggle_array, leakage_model)
        corr_toggles = compute_correlation(toggle_array, leakage_model)
        # 3) Store results
        max_val = np.max(np.abs(corr_toggles))
        best_idx = np.argmax(np.abs(corr_toggles))
        leakage_sum_result.append((sig_name, leakage_sum))
        toggle_corr_results.append((sig_name, corr_toggles, max_val, best_idx))

    # Sort by max absolute correlation
    toggle_corr_results.sort(key=lambda x: x[2], reverse=True)

    ############# (E) Plot the correlation vs. clock for each signal ###############
    # WARNING: if you have hundreds of signals, this means hundreds of plots!
    # You may want to limit to top 10 or top 20 signals. Example:
    # for i, (sig_name, corr_toggles, max_val, best_idx) in enumerate(toggle_corr_results[:10]):
    #     ...
    # If you'd like, also print top results in console:
    print("\n=== Top Toggle-Correlated Signals ===")
    for i, (sig_name, corr_arr, max_val, best_idx) in enumerate(toggle_corr_results[:20]):
        print(f"{i+1:2d}) {sig_name}: max abs corr={max_val:.4f} at clock={best_idx}")
    threshold = 0.2

    start_ps = 268350000
    clock_step_ps = 50000

    plt.figure()

    for (sig_name, corr_toggles, max_val, best_idx) in toggle_corr_results:
        num_clocks = corr_toggles.shape[0]

        # Build time axis in ps
        time_ps = start_ps + clock_step_ps * np.arange(num_clocks)

        # Check if this signal's max correlation exceeds the threshold
        if max_val > threshold:
            label_str = sig_name
            if sig_name == "u_top_gen_regfile_ff.register_file_i_rf_reg_q[13] [0]":
                label_str = "register_file_i_rf_reg_q[13] [0]"
        else:
            label_str = None

        plt.plot(time_ps, corr_toggles, label=label_str)

    plt.title("All Signals Toggle Correlation with AES stat bit 0 (vs. Time)")
    plt.xlabel("Time (ps)")
    plt.ylabel("Correlation")
    plt.legend()
    plt.savefig("Toggle2_correlation.pdf", bbox_inches="tight", transparent=True)
    plt.savefig("Toggle2_correlation.svg", bbox_inches="tight", transparent=True)
    plt.savefig("Toggle2_correlation.png", bbox_inches="tight", transparent=True)
    # Only signals labeled above will appear in the legend

    plt.show()


















In [ ]:
import numpy as np
import os
import re
from tqdm import tqdm
from scipy.stats import pearsonr, ConstantInputWarning
import warnings
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed

###############################################################################
# Additional: Example s-box + HD model
###############################################################################
sbox = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]

###############################################################################
# Single-bit HD model extraction
###############################################################################
def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=0):
    leakage_model = []
    for pt in plaintexts:
        sbox_in = pt ^ key_byte
        bit_in = (sbox_in >> selected_bit) & 0x1
        sbox_out = sbox[sbox_in]
        bit_out = (sbox_out >> selected_bit) & 0x1
        hd_bit = bit_in ^ bit_out
        leakage_model.append(1 if hd_bit == 1 else -1)
    return np.array(leakage_model, dtype=np.float64)


###############################################################################
# (1) Single-file Parser: Count toggles each clock cycle
###############################################################################
def parse_vcd_total_toggles(vcd_file, clock_signal_name="CLK"):
    """
    Parse the given VCD file. Track signals' old values and accumulate
    the total number of bit toggles for each clock cycle.

    - A 'cycle' is from one rising edge of the clock_signal_name to the next.
    - On each signal change, add the bit-level Hamming distance to toggles_count.
    - On a rising clock edge, push toggles_count into toggles_per_cycle[] and reset.

    Returns:
      toggles_per_cycle: list[int], length = number of cycles in this VCD
    """

    clock_id = None
    signal_map = {}     # symbol_id -> signal_name
    signal_vals= {}     # symbol_id -> integer-coded old value
    toggles_count = 0
    toggles_per_cycle = []

    def hamming_distance_int(old_val, new_val):
        return bin(old_val ^ new_val).count("1")

    in_dumpvars  = False
    current_time = 0

    with open(vcd_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith("$var"):
                parts = line.split()
                if len(parts) >= 5:
                    sym_id   = parts[3]
                    name_tokens = parts[4:-1]
                    sig_name = " ".join(name_tokens).strip()
                    signal_map[sym_id] = sig_name
                    signal_vals[sym_id] = 0
                    if sig_name == clock_signal_name:
                        clock_id = sym_id
                continue

            if line.startswith("$dumpvars"):
                in_dumpvars = True
                continue
            if line.startswith("$end") and in_dumpvars:
                in_dumpvars = False
                continue

            if line.startswith("#"):
                try:
                    current_time = int(line[1:])
                except ValueError:
                    current_time = 0
                continue

            # Single-bit change
            m_single = re.match(r"([01xXzZ])(\S+)", line)
            if m_single:
                val_str, sym_id = m_single.groups()
                if sym_id not in signal_vals:
                    continue

                old_val = signal_vals[sym_id]
                val_str = val_str.lower()
                if val_str in ('x','z'):
                    new_val = 0
                else:
                    new_val = int(val_str, 2)

                if new_val != old_val:
                    dist = hamming_distance_int(old_val, new_val)
                    toggles_count += dist
                    signal_vals[sym_id] = new_val

                    if sym_id == clock_id:
                        # Check for rising edge: old_val=0, new_val=1
                        #if old_val == 0 and new_val == 1:
                        toggles_per_cycle.append(toggles_count)
                        toggles_count = 0
                continue

            # Multi-bit change
            m_bus = re.match(r"b([01xXzZ]+)\s+(\S+)", line)
            if m_bus:
                bits_str, sym_id = m_bus.groups()
                if sym_id not in signal_vals:
                    continue
                old_val = signal_vals[sym_id]
                bits_str = bits_str.lower().replace('x','0').replace('z','0')
                new_val = int(bits_str, 2)
                if new_val != old_val:
                    dist = hamming_distance_int(old_val, new_val)
                    toggles_count += dist
                    signal_vals[sym_id] = new_val

                    if sym_id == clock_id:
                        #if old_val != new_val: #old_val == 0 and new_val == 1:
                        toggles_per_cycle.append(toggles_count)
                        toggles_count = 0
                continue

    return toggles_per_cycle


###############################################################################
# (2) Parallel parse all VCDs -> Build 2D array of toggles
###############################################################################
def build_toggle_traces(folder, clock_signal, total_files=600, mxworker=4):
    """
    For i in [1..total_files], parse "vcd{i}.vcd" from `folder` in parallel.
    Return a 2D NumPy array of shape (total_files, max_cycles),
    where row i is the toggles-per-cycle from vcd{i}.vcd, padded with 0.
    """
    filepaths = [os.path.join(folder, f"vcd{i}.vcd") for i in range(1, total_files+1)]
    all_results = [None]*total_files
    max_len = 0

    # Parse in parallel
    with ProcessPoolExecutor(max_workers=mxworker) as executor:
        futures_map = {}
        for i, fp in enumerate(filepaths):
            fut = executor.submit(parse_vcd_total_toggles, fp, clock_signal)
            futures_map[fut] = i

        for fut in tqdm(as_completed(futures_map), total=total_files, desc="Parsing VCDs"):
            idx = futures_map[fut]
            try:
                toggles_list = fut.result()
            except Exception as e:
                print(f"[ERROR] Parsing file {filepaths[idx]}: {e}")
                toggles_list = []
            all_results[idx] = toggles_list
            if len(toggles_list) > max_len:
                max_len = len(toggles_list)

    # Now unify into 2D array
    toggles_2d = np.zeros((total_files, max_len), dtype=np.float64)
    for i in range(total_files):
        row_len = len(all_results[i])
        toggles_2d[i, :row_len] = all_results[i]

    return toggles_2d


###############################################################################
# (3) Pearson correlation
###############################################################################
def compute_pearson_correlation(toggle_traces, leakage_model):
    """
    toggle_traces: shape (num_traces, num_cycles)
    leakage_model: shape (num_traces,)
    returns correlation array of shape (num_cycles,)
    """
    num_traces, num_cycles = toggle_traces.shape
    corr_over_time = np.zeros(num_cycles, dtype=float)

    for t in range(num_cycles):
        column = toggle_traces[:, t]
        if np.all(column == column[0]) or np.all(leakage_model == leakage_model[0]):
            corr_over_time[t] = 0.0
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=ConstantInputWarning)
                r, _ = pearsonr(column, leakage_model)
                corr_over_time[t] = r

    return corr_over_time


###############################################################################
# (4) Example usage
###############################################################################
if __name__ == "__main__":
    import time

    time_start = time.time()

    # Configuration
    vcd_folder     = "/media/abish/Extreme Pro/vcdfull600/vcds"
    path           = "/media/abish/Extreme Pro/vcdfull600/p"
    clock_name     = "SYSCLK_P"
    total_vcds     = 600
    mxworker       = 15   # how many parallel processes to parse files

    # 1) Load plaintexts and build leakage model
    txt_path = os.path.join(path, "plaintexts.txt")
    hex_arrays = []
    with open(txt_path, "r") as file:
        for line in file:
            parts = line.strip().split(":")
            if len(parts) == 2:
                hex_values = parts[1].strip().split()
                hex_arrays.append(hex_values)

    # Convert hex plaintext lines to 2D array
    textin_array = []
    for row in hex_arrays:
        textin_array.append([int(x, 16) for x in row])
    textin_array = np.array(textin_array, dtype=np.uint8)
    print("textin_array shape:", textin_array.shape)

    # Example key guess
    corkey = [0x2b,0x7e,0x15,0x16,0x28,0xae,0xd2,0xa6,
              0xab,0xf7,0x15,0x88,0x09,0xcf,0x4f,0x3c]
    selected_byte = 0
    chosen_bit    = 0  # LSB
    key_guess     = corkey[selected_byte]

    plaintext_for_each_trace = textin_array[:, selected_byte]
    leakage_model = extract_leakage_model_hd_bit(
        plaintexts=plaintext_for_each_trace,
        key_byte=key_guess,
        selected_bit=chosen_bit
    )
    print(f"Built HD model for {total_vcds} traces (bit={chosen_bit}).")

    # 2) Build 2D toggles array (with parallel parse)
    print("1) Parsing all VCDs and building toggles 2D array ...")
    toggle_traces = build_toggle_traces(
        folder=vcd_folder,
        clock_signal=clock_name,
        total_files=total_vcds,
        mxworker=mxworker
    )
    print(f"toggle_traces shape = {toggle_traces.shape}")

    # 3) Compute correlation vs. leakage model
    print("2) Compute correlation vs. leakage model ...")
    corr_time = compute_pearson_correlation(toggle_traces, leakage_model)

    # 4) Plot result
    plt.figure()
    plt.plot(corr_time, label="Correlation(toggle_count, model)")
    plt.title("Total Toggles (All Signals) vs. Model (Parallel Parsing)")
    plt.xlabel("Cycle index")
    plt.ylabel("Correlation")
    plt.legend()
    plt.savefig("total_toggles_correlation.pdf", bbox_inches="tight", transparent=True)
    plt.savefig("total_toggles_correlation.png", bbox_inches="tight", transparent=True)
    plt.show()

    time_end = time.time()
    print(f"Done. Elapsed time = {time_end - time_start:.2f}s")
###this just count overal toggels for finding LTI

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, time
from scipy.stats import pearsonr, ConstantInputWarning
import warnings
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

###############################################################################
# Example S-box + HD function (same as before)
###############################################################################
sbox = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
]

def extract_leakage_model_hd_bit(plaintexts, key_byte, selected_bit=0):
    """
    For each plaintext, compute HD between sbox_in vs. sbox_out at selected bit.
    """
    leakage_model = []
    for pt in plaintexts:
        sbox_in = pt ^ key_byte
        bit_in = (sbox_in >> selected_bit) & 0x1
        sbox_out = sbox[sbox_in]
        bit_out = (sbox_out >> selected_bit) & 0x1
        hd_bit = bit_in ^ bit_out
        leakage_model.append(1 if hd_bit == 1 else -1)
    return np.array(leakage_model, dtype=np.float64)


###############################################################################
# Pearson correlation function
###############################################################################
def compute_pearson_correlation(toggle_traces, leakage_model):
    """
    toggle_traces: shape (num_traces, num_cycles)
    leakage_model: shape (num_traces,)
    returns correlation array of shape (num_cycles,)
    """
    import warnings
    from scipy.stats import pearsonr, ConstantInputWarning

    num_traces, num_cycles = toggle_traces.shape
    corr_over_time = np.zeros(num_cycles, dtype=float)

    for t in range(num_cycles):
        column = toggle_traces[:, t]
        if np.all(column == column[0]) or np.all(leakage_model == leakage_model[0]):
            corr_over_time[t] = 0.0
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=ConstantInputWarning)
                r, _ = pearsonr(column, leakage_model)
                corr_over_time[t] = r

    return corr_over_time

###############################################################################
# MAIN
###############################################################################
if __name__ == "__main__":
    time_start = time.time()
    corkey = [0x2b,0x7e,0x15,0x16,0x28,0xae,0xd2,0xa6,
              0xab,0xf7,0x15,0x88,0x09,0xcf,0x4f,0x3c]

    num_bytes = 16
    num_bits  = 8
    num_cycles= toggle_traces.shape[1]

    corr_by_byte_and_bit = np.zeros((num_bytes, num_bits, num_cycles), dtype=float)

    for b in range(num_bytes):
        key_guess = corkey[b]
        # plaintext_for_this_byte has shape (600,)
        plaintext_for_this_byte = textin_array[:, b]

        for bit_i in range(num_bits):
            # Build HD model for [byte b, bit bit_i]
            leakage_model = extract_leakage_model_hd_bit(
                plaintext_for_this_byte,
                key_byte=key_guess,  # use the exact param name
                selected_bit=bit_i
            )
            # Correlate
            corr_t = compute_pearson_correlation(toggle_traces, leakage_model)
            corr_by_byte_and_bit[b, bit_i, :] = corr_t

    # 4) Plot all 128 lines (16 bytes × 8 bits) on one figure
    plt.figure()
    for b in range(num_bytes):
        for bit_i in range(num_bits):
            label_str = f"Byte{b},Bit{bit_i}"
            plt.plot(corr_by_byte_and_bit[b, bit_i, :], label=label_str)

    plt.title("Correlation for All 16 Bytes × 8 Bits")
    plt.xlabel("Cycle index")
    plt.ylabel("Correlation")
    #plt.legend()  # be aware this might be huge with 128 items
    plt.savefig("all_128_correlations.png", bbox_inches="tight", transparent=True)
    plt.show()


In [ ]:
import matplotlib.patches as patches

# Convert cycle index to time (each cycle = 50000 ps)
threshold = 0.188
indices_above_threshold = np.argwhere(np.abs(corr_by_byte_and_bit) > threshold)

if indices_above_threshold.size > 0:
    min_idx = indices_above_threshold[np.argmin(indices_above_threshold[:, 2])]
    max_idx = indices_above_threshold[np.argmax(indices_above_threshold[:, 2])]

    b1, bit1, cycle1 = min_idx
    b2, bit2, cycle2 = max_idx

    time_val1 = cycle1 * 50000
    time_val2 = cycle2 * 50000

    corr_val1 = corr_by_byte_and_bit[b1, bit1, cycle1]
    corr_val2 = corr_by_byte_and_bit[b2, bit2, cycle2]

    # Compute rectangle position and size
    rect_x = min(time_val1, time_val2)-0.015
    rect_y = min(corr_val1, corr_val2)-.01
    rect_width = abs(time_val2 - time_val1)+0.08
    rect_height = abs(corr_val2 - corr_val1)+0.04

    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    for b in range(num_bytes):
        for bit_i in range(num_bits):
            ax.plot(time_axis, corr_by_byte_and_bit[b, bit_i, :], alpha=0.5)

    # Draw rectangle
    rect = patches.Rectangle(
        (rect_x, rect_y),
        rect_width,
        rect_height,
        linewidth=2,
        edgecolor='red',
        facecolor='red',
        alpha=0.3,
        zorder=10
    )
    ax.add_patch(rect)

    ax.set_xlabel("Time (ps)", fontweight='bold')
    ax.set_ylabel("Correlation", fontweight='bold')
    ax.grid(True)
    plt.tight_layout()
    plt.savefig("highlighted_rectangle_fixed.png", bbox_inches="tight", transparent=True)
    plt.savefig("highlighted_rectangle_fixed.pdf", bbox_inches="tight", transparent=True)
    plt.savefig("highlighted_rectangle_fixed.svg", bbox_inches="tight", transparent=True)
    plt.show()

else:
    print("No correlation values greater than 0.192 found.")


In [ ]:
import matplotlib.patches as patches

# Convert cycle index to time (each cycle = 50000 ps)
threshold = 0.188
indices_above_threshold = np.argwhere(np.abs(corr_by_byte_and_bit) > threshold)

if indices_above_threshold.size > 0:
    min_idx = indices_above_threshold[np.argmin(indices_above_threshold[:, 2])]
    max_idx = indices_above_threshold[np.argmax(indices_above_threshold[:, 2])]

    b1, bit1, cycle1 = min_idx
    b2, bit2, cycle2 = max_idx

    time_val1 = cycle1 * 50000
    time_val2 = cycle2 * 50000

    corr_val1 = corr_by_byte_and_bit[b1, bit1, cycle1]
    corr_val2 = corr_by_byte_and_bit[b2, bit2, cycle2]

    # Compute rectangle position and size
    rect_x = min(time_val1, time_val2) - 0.015
    rect_y = min(corr_val1, corr_val2) - 0.01
    rect_width = abs(time_val2 - time_val1) + 0.08
    rect_height = abs(corr_val2 - corr_val1) + 0.04

    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    for b in range(num_bytes):
        for bit_i in range(num_bits):
            ax.plot(time_axis, corr_by_byte_and_bit[b, bit_i, :], alpha=0.5)

    # Draw rectangle
    rect = patches.Rectangle(
        (rect_x, rect_y),
        rect_width,
        rect_height,
        linewidth=2,
        edgecolor='red',
        facecolor='red',
        alpha=0.3,
        zorder=10
    )
    ax.add_patch(rect)

    # Set axis labels and formatting
    ax.set_xlabel("Time (ps)", fontsize=14, fontweight='bold')
    ax.set_ylabel("Correlation", fontsize=14, fontweight='bold')

    # Make axis ticks bold and larger
    ax.tick_params(axis='both', which='major', labelsize=12)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')

    ax.grid(True)
    plt.tight_layout()

    # Save plots
    plt.savefig("highlighted_rectangle_fixed.png", bbox_inches="tight", transparent=True)
    plt.savefig("highlighted_rectangle_fixed.pdf", bbox_inches="tight", transparent=True)
    plt.savefig("highlighted_rectangle_fixed.svg", bbox_inches="tight", transparent=True)
    plt.show()

else:
    print("No correlation values greater than 0.192 found.")
